# Full Atlas-Free ALE3DCNN Pipeline

This notebook is the single playbook for the atlas-free CNN workflow.

Pipeline:

```text
Stage 0: pack PubMed + Nilearn + NeuroVault into HF-friendly CNN tensors/text pairs
Stage 1: train atlas-free CNN autoencoder
Stage 1 eval: reconstruction capability on train/val/test
Stage 2: initialize encoder from the autoencoder encoder
Stage 3: contrastive fine-tuning for brain-to-text retrieval
Stage 3 eval: brain-to-text and text-to-brain retrieval metrics
Stage 4: train a text-to-brain projection head on train only and tune on val
Stage 5: text-to-brain generation evaluation on held-out test/network data
```

**Hard rule:** the projection head is trained before generation evaluation. Held-out test-set text-to-brain generation is evaluation, not training: train on train, select/tune on val, and evaluate once on test.

## Colab Setup

Run this cell first in Colab. It clones the repo, installs the package, and switches the working directory to the repo root. The next setup cell mounts Google Drive and saves model checkpoints, plots, evaluation files, and summaries under `MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline`. If you are already running inside a local checkout, it leaves your checkout alone.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_REF = 'neurovlm_gnn'
COLAB_REPO = Path('/content/neurovlm')
IN_COLAB = Path('/content').exists() and 'google.colab' in sys.modules

def _has_repo_root(path):
    return (Path(path) / 'pyproject.toml').exists() and (Path(path) / 'src/neurovlm').exists()

if IN_COLAB:
    if not _has_repo_root(COLAB_REPO):
        subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(COLAB_REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(COLAB_REPO), 'fetch', 'origin', REPO_REF], check=False)
        subprocess.run(['git', '-C', str(COLAB_REPO), 'checkout', REPO_REF], check=False)
        subprocess.run(['git', '-C', str(COLAB_REPO), 'pull', '--ff-only', 'origin', REPO_REF], check=False)
    os.chdir(COLAB_REPO)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
else:
    print('Not in Colab, using the current local environment.')

# `pip install -e .` installs src/neurovlm, while this experiment package lives at repo root.
REPO_ROOT = Path.cwd()
for import_root in [REPO_ROOT, REPO_ROOT / 'src']:
    import_root = str(import_root)
    if import_root not in sys.path:
        sys.path.insert(0, import_root)

git_head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], text=True, capture_output=True)
print('cwd:', Path.cwd())
print('repo ref:', REPO_REF)
print('git head:', git_head.stdout.strip() or 'unknown')
print('repo-root experiment package exists:', (REPO_ROOT / 'experiments/3dcnn/atlas_free_cnn/training').exists())


cwd: /content/neurovlm
repo ref: neurovlm_gnn
git head: 64d5248
repo-root experiment package exists: True


In [2]:

from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import torch


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    return start


ROOT = find_repo_root()
for import_root in [ROOT, ROOT / 'src', ROOT / 'experiments/3dcnn']:
    import_root = str(import_root)
    if import_root not in sys.path:
        sys.path.insert(0, import_root)

CACHE = ROOT / 'experiments/3dcnn/atlas_free_cnn/cache'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
IN_COLAB = Path('/content').exists() and 'google.colab' in sys.modules

drive_mounted = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_mounted = Path('/content/drive/MyDrive').exists()
    except Exception as exc:
        print('Google Drive mount skipped:', repr(exc))

if drive_mounted:
    DRIVE_ROOT = Path('/content/drive/MyDrive/neurovlm')
    RUNS_DIR = DRIVE_ROOT / 'runs_atlas_free_cnn_full_pipeline'
    EVAL_RESOURCE_DIR = Path('/content/drive/MyDrive/neurovlm_evaluation_resources')
    OUT = RUNS_DIR / f'full_atlas_free_cnn_{RUN_STAMP}'
else:
    DRIVE_ROOT = ROOT / 'experiments/3dcnn/outputs'
    RUNS_DIR = DRIVE_ROOT / 'runs_atlas_free_cnn_full_pipeline'
    EVAL_RESOURCE_DIR = ROOT / 'experiments/evaluation_resources'
    OUT = RUNS_DIR / f'full_atlas_free_cnn_{RUN_STAMP}'

for path_ in [DRIVE_ROOT, RUNS_DIR, EVAL_RESOURCE_DIR, OUT]:
    path_.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGET_SHAPE = (36, 45, 38)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TEST_IS_EVALUATION_ONLY = True

# Stage 1 defaults now reproduce the old good PubMed-only autoencoder first.
RUN_MODE = globals().get('RUN_MODE', 'full')  # 'full' or 'smoke'
DATA_MODE = globals().get('DATA_MODE', 'mixed')  # 'pubmed_only' or 'mixed'
AE_TRAINING_RECIPE = globals().get('AE_TRAINING_RECIPE', 'previous_good_compatible')
if RUN_MODE not in {'full', 'smoke'}:
    raise ValueError("RUN_MODE must be 'full' or 'smoke'")
if DATA_MODE not in {'pubmed_only', 'mixed'}:
    raise ValueError("DATA_MODE must be 'pubmed_only' or 'mixed'")
if AE_TRAINING_RECIPE != 'previous_good_compatible':
    raise ValueError("Only AE_TRAINING_RECIPE='previous_good_compatible' remains in the clean 3D CNN pipeline.")

run_config = {
    'device': str(DEVICE),
    'target_shape': TARGET_SHAPE,
    'test_is_eval_only': TEST_IS_EVALUATION_ONLY,
    'run_mode': RUN_MODE,
    'data_mode': DATA_MODE,
    'ae_training_recipe': AE_TRAINING_RECIPE,
    'root': str(ROOT),
    'drive_root': str(DRIVE_ROOT),
    'runs_dir': str(RUNS_DIR),
    'eval_resource_dir': str(EVAL_RESOURCE_DIR),
    'out': str(OUT),
    'run_stamp': RUN_STAMP,
}
(OUT / 'run_config.json').write_text(json.dumps(run_config, indent=2))
print(json.dumps(run_config, indent=2, default=str))


Mounted at /content/drive
{
  "device": "cuda",
  "target_shape": [
    36,
    45,
    38
  ],
  "test_is_eval_only": true,
  "run_mode": "full",
  "data_mode": "mixed",
  "ae_training_recipe": "previous_good_compatible",
  "root": "/content/neurovlm",
  "drive_root": "/content/drive/MyDrive/neurovlm",
  "runs_dir": "/content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline",
  "eval_resource_dir": "/content/drive/MyDrive/neurovlm_evaluation_resources",
  "out": "/content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline/full_atlas_free_cnn_20260528_053320",
  "run_stamp": "20260528_053320"
}


## Stage 0: Pack Data For Local/Hugging Face Use

Run this after building PubMed/Nilearn split JSONL files and the staged NeuroVault collector. NeuroVault defaults to deterministic train/val/test splitting, so it does not all leak into train.

The packed output is intended for Hugging Face upload:

```text
atlas_free_cnn_volumes.pt
atlas_free_cnn_rows.parquet
atlas_free_cnn_text_pairs.parquet
atlas_free_cnn_manifest.json
```

The text-pair parquet includes `is_primary_text`. Primary text is selected by metadata specificity, not by model similarity. For this single-text 3DCNN experiment the intended primary policy is: PubMed title + summary/abstract-style text; NeuroVault image title + description or collection title + description, falling back to task/contrast label; Nilearn atlas/network label.

In [3]:
# Local packing command. Run in a terminal/local checkout, not in Colab.
# This creates the files that should be uploaded to Hugging Face.
# !.conda/bin/python -m atlas_free_cnn.data_building.pack_atlas_free_cnn_dataset \
#   --jsonl experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits/train.jsonl \
#           experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits/val.jsonl \
#           experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits/test.jsonl \
#   --neurovault-dir experiments/3dcnn/atlas_free_cnn/cache/neurovault_diverse_5k \
#   --neurovault-split auto \
#   --neurovault-max-per-collection 50 \
#   --strong-neurovault-only \
#   --output-dir experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn

PACKED_DIR = CACHE / 'hf_atlas_free_cnn'
LOCAL_VOLUME_PT = PACKED_DIR / 'atlas_free_cnn_volumes.pt'
LOCAL_ROWS = PACKED_DIR / 'atlas_free_cnn_rows.parquet'
LOCAL_TEXT_PAIRS = PACKED_DIR / 'atlas_free_cnn_text_pairs.parquet'
LOCAL_MANIFEST = PACKED_DIR / 'atlas_free_cnn_manifest.json'

if LOCAL_MANIFEST.exists():
    print(json.loads(LOCAL_MANIFEST.read_text()))
else:
    print('Local packed dataset not found. In Colab this is expected; use Hugging Face loading below.')
    print('Missing:', PACKED_DIR)


Local packed dataset not found. In Colab this is expected; use Hugging Face loading below.
Missing: /content/neurovlm/experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn


## Colab/Hugging Face Loading

Colab should load the packed dataset from Hugging Face. Local files under `/content/experiments/3dcnn/atlas_free_cnn/cache/...` are not expected to exist unless you manually copied them there.

Expected HF dataset repo:

```text
neurovlm/atlas_free_cnn_dataset
```


In [4]:
HF_DATASET_REPO = 'neurovlm/atlas_free_cnn_dataset'
USE_HF_DATA = True  # Colab default. Set False only in a local checkout with packed files present.

def _local_packed_files_exist():
    return all(p.exists() for p in [LOCAL_VOLUME_PT, LOCAL_ROWS, LOCAL_TEXT_PAIRS, LOCAL_MANIFEST])

def load_packed_dataset(prefer_hf=True, repo_id=HF_DATASET_REPO):
    if prefer_hf:
        try:
            from huggingface_hub import hf_hub_download
            volume_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_volumes.pt', repo_type='dataset')
            rows_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_rows.parquet', repo_type='dataset')
            pairs_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_text_pairs.parquet', repo_type='dataset')
            manifest_path = hf_hub_download(repo_id=repo_id, filename='atlas_free_cnn_manifest.json', repo_type='dataset')
            print('Loaded packed atlas-free CNN dataset from Hugging Face:', repo_id)
            payload = torch.load(volume_path, map_location='cpu', weights_only=False)
            return {
                **payload,
                'rows': pd.read_parquet(rows_path),
                'text_pairs': pd.read_parquet(pairs_path),
                'manifest': json.loads(Path(manifest_path).read_text()),
            }
        except Exception as exc:
            print('HF load failed:', repr(exc))
            print('If this is a private repo, run `huggingface-cli login` or set an HF token.')

    if not _local_packed_files_exist():
        missing = [str(p) for p in [LOCAL_VOLUME_PT, LOCAL_ROWS, LOCAL_TEXT_PAIRS, LOCAL_MANIFEST] if not p.exists()]
        raise FileNotFoundError('Packed dataset not found locally. Use USE_HF_DATA=True in Colab, or upload/copy packed files. Missing: ' + '; '.join(missing))

    print('Loaded packed atlas-free CNN dataset from local files:', PACKED_DIR)
    payload = torch.load(LOCAL_VOLUME_PT, map_location='cpu', weights_only=False)
    return {
        **payload,
        'rows': pd.read_parquet(LOCAL_ROWS),
        'text_pairs': pd.read_parquet(LOCAL_TEXT_PAIRS),
        'manifest': json.loads(LOCAL_MANIFEST.read_text()),
    }

packed = load_packed_dataset(prefer_hf=USE_HF_DATA)
volumes = packed['volumes'].float()
rows = packed['rows'].copy()
text_pairs = packed['text_pairs'].copy()
print(volumes.shape, rows.shape, text_pairs.shape)
print(rows['split'].value_counts())
print(rows['source'].value_counts().head(20))


atlas_free_cnn_volumes.pt:   0%|          | 0.00/4.16G [00:00<?, ?B/s]

atlas_free_cnn_rows.parquet:   0%|          | 0.00/15.7M [00:00<?, ?B/s]

atlas_free_cnn_text_pairs.parquet:   0%|          | 0.00/22.8M [00:00<?, ?B/s]

atlas_free_cnn_manifest.json: 0.00B [00:00, ?B/s]

Loaded packed atlas-free CNN dataset from Hugging Face: neurovlm/atlas_free_cnn_dataset
torch.Size([33657, 1, 36, 45, 38]) (33657, 15) (103800, 10)
split
train    26944
val       3366
test      3347
Name: count, dtype: int64
source
pubmed                                30658
neurovault                             2202
nilearn:schaefer_2018                   400
nilearn:basc_064                         64
nilearn:difumo_64                        64
nilearn:juelich_probabilistic            62
nilearn:juelich                          62
nilearn:harvard_oxford_cortical          48
nilearn:msdl                             39
nilearn:harvard_oxford_subcortical       21
nilearn:smith_2009                       20
nilearn:yeo_2011                         17
Name: count, dtype: int64


## Materialize JSONL Splits For Existing Trainers

The current training scripts expect JSONL rows with `tensor_path`, `tensor_index`, and `positive_texts`. For this 3DCNN experiment we are **not** doing multi-positive text yet, so each map gets exactly one primary text target selected by metadata priority: PubMed title + summary/abstract-style text; NeuroVault title + description if available, otherwise task label; Nilearn atlas/network label. The test split is written for evaluation loaders only.

In [5]:

def canonical_source(source):
    source = str(source or '').lower()
    if source == 'pubmed' or source.startswith('pubmed'):
        return 'pubmed'
    if source == 'neurovault' or source.startswith('neurovault'):
        return 'neurovault'
    if source.startswith('nilearn'):
        return 'nilearn'
    return source or 'unknown'


def source_detail(row):
    source = str(row.get('source', '') or '')
    if source.startswith('nilearn:'):
        return source
    for key in ['atlas_name', 'neurovault_collection_key', 'collection_id']:
        value = row.get(key)
        if value is not None and str(value) not in {'', 'nan', 'None'}:
            return f'{canonical_source(source)}:{value}'
    return canonical_source(source)


def _filter_rows_for_data_mode(rows_df, data_mode):
    rows_df = rows_df.copy()
    rows_df['_canonical_source'] = rows_df['source'].map(canonical_source)
    if data_mode == 'pubmed_only':
        return rows_df[rows_df['_canonical_source'] == 'pubmed'].drop(columns=['_canonical_source'])
    if data_mode == 'mixed':
        return rows_df[rows_df['_canonical_source'].isin(['pubmed', 'neurovault', 'nilearn'])].drop(columns=['_canonical_source'])
    raise ValueError(data_mode)


def materialize_primary_jsonl_splits_from_packed(packed, out_dir, *, data_mode):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    tensor_path = out_dir / 'atlas_free_cnn_volumes.pt'
    torch.save({'volumes': packed['volumes'], 'map_ids': packed.get('map_ids')}, tensor_path)
    rows_df = _filter_rows_for_data_mode(packed['rows'], data_mode)
    pairs_df = packed['text_pairs'].copy()
    pairs_df = pairs_df[pairs_df['is_primary_text']].copy()
    written = {}
    for split in ['train', 'val', 'test']:
        split_rows = rows_df[rows_df['split'] == split]
        path = out_dir / f'{split}.jsonl'
        with path.open('w') as f:
            for _, row in split_rows.iterrows():
                positives = pairs_df[pairs_df.map_id == row.map_id].sort_values('pair_rank').head(1)
                record = row.to_dict()
                record.update({
                    'tensor_path': str(tensor_path),
                    'tensor_index': int(row.tensor_index),
                    'canonical_source': canonical_source(row.get('source')),
                    'source_detail': source_detail(row),
                    'positive_texts': positives[['text','term','category','source','weight','reliability']].to_dict('records'),
                    'primary_text_only': True,
                })
                f.write(json.dumps(record, default=str) + '\n')
        written[split] = path
    return written


MATERIALIZED_ROOT = CACHE / 'hf_atlas_free_cnn/materialized_primary_jsonl'
ALL_SPLIT_JSONL = {
    mode: materialize_primary_jsonl_splits_from_packed(packed, MATERIALIZED_ROOT / mode, data_mode=mode)
    for mode in ['pubmed_only', 'mixed']
}
SPLIT_JSONL = ALL_SPLIT_JSONL[DATA_MODE]

rows = rows.copy()
rows['canonical_source'] = rows['source'].map(canonical_source)
rows['source_detail'] = rows.apply(source_detail, axis=1)
source_counts = rows.groupby(['split', 'canonical_source']).size().rename('n').reset_index()
source_detail_counts = rows.groupby(['split', 'source_detail']).size().rename('n').reset_index()
source_counts.to_csv(OUT / 'packed_source_counts_by_split.csv', index=False)
source_detail_counts.to_csv(OUT / 'packed_source_detail_counts_by_split.csv', index=False)
print('Active RUN_MODE:', RUN_MODE)
print('Active DATA_MODE:', DATA_MODE)
print('Active JSONL splits:', SPLIT_JSONL)
print('Source counts by split:')
display(source_counts)


def _sample_flat_values(x, max_values=500_000, seed=SEED):
    flat = x.flatten()
    if flat.numel() <= max_values:
        return flat.float().cpu()
    g = torch.Generator(device='cpu')
    g.manual_seed(int(seed))
    idx = torch.randperm(flat.numel(), generator=g)[:max_values]
    return flat.cpu()[idx].float()


def volume_stats_for_indices(indices, *, split, source, detail=None, chunk_size=128, percentile_sample_voxels=500_000):
    indices = [int(i) for i in indices]
    if not indices:
        return None
    total_voxels = 0
    nonzero_voxels = 0
    value_sum = 0.0
    value_sumsq = 0.0
    min_value = float('inf')
    max_value = -float('inf')
    empty_maps = 0
    nearly_empty_maps = 0
    samples = []
    sample_budget = int(percentile_sample_voxels)
    for start in range(0, len(indices), chunk_size):
        batch_idx = indices[start:start + chunk_size]
        x = volumes[batch_idx].float().cpu()
        flat = x.flatten(1)
        total_voxels += int(x.numel())
        nonzero_voxels += int((x != 0).sum().item())
        value_sum += float(x.sum().item())
        value_sumsq += float(x.pow(2).sum().item())
        min_value = min(min_value, float(x.min().item()))
        max_value = max(max_value, float(x.max().item()))
        per_map_nonzero = (flat != 0).float().mean(dim=1)
        per_map_max = flat.max(dim=1).values
        empty_maps += int((per_map_max <= 1e-8).sum().item())
        nearly_empty_maps += int(((per_map_nonzero < 1e-5) | (per_map_max <= 1e-6)).sum().item())
        if sample_budget > 0:
            sample = _sample_flat_values(x, max_values=min(sample_budget, 100_000), seed=SEED + start)
            samples.append(sample)
            sample_budget -= int(sample.numel())
    mean = value_sum / max(1, total_voxels)
    var = max(0.0, value_sumsq / max(1, total_voxels) - mean * mean)
    sample_values = torch.cat(samples) if samples else torch.tensor([], dtype=torch.float32)
    qs = torch.quantile(sample_values, torch.tensor([0.95, 0.99, 0.995])) if sample_values.numel() else torch.tensor([float('nan')] * 3)
    return {
        'split': split,
        'source': source,
        'source_detail': detail or source,
        'n_maps': len(indices),
        'min': min_value,
        'max': max_value,
        'mean': mean,
        'std': var ** 0.5,
        'fraction_nonzero_voxels': nonzero_voxels / max(1, total_voxels),
        'p95_sampled': float(qs[0].item()),
        'p99_sampled': float(qs[1].item()),
        'p995_sampled': float(qs[2].item()),
        'empty_maps': empty_maps,
        'nearly_empty_maps': nearly_empty_maps,
        'targets_in_0_1': bool(min_value >= -1e-6 and max_value <= 1.0 + 1e-6),
        'percentile_sample_voxels': int(sample_values.numel()),
    }


stats_rows = []
for split, split_rows in rows.groupby('split'):
    stats_rows.append(volume_stats_for_indices(split_rows.tensor_index, split=split, source='all', detail='all'))
    for source, source_rows in split_rows.groupby('canonical_source'):
        stats_rows.append(volume_stats_for_indices(source_rows.tensor_index, split=split, source=source, detail=source))
    atlas_rows = split_rows[split_rows['canonical_source'] == 'nilearn']
    for detail, detail_rows in atlas_rows.groupby('source_detail'):
        stats_rows.append(volume_stats_for_indices(detail_rows.tensor_index, split=split, source='nilearn', detail=detail))
preprocessing_stats = pd.DataFrame([r for r in stats_rows if r is not None])
preprocessing_stats.to_csv(OUT / 'input_volume_stats_by_split_source.csv', index=False)
print('Saved preprocessing/input stats:', OUT / 'input_volume_stats_by_split_source.csv')
display(preprocessing_stats[preprocessing_stats['source_detail'].isin(['all', 'pubmed', 'neurovault', 'nilearn'])].sort_values(['split', 'source_detail']))


Active RUN_MODE: full
Active DATA_MODE: mixed
Active JSONL splits: {'train': PosixPath('/content/neurovlm/experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn/materialized_primary_jsonl/mixed/train.jsonl'), 'val': PosixPath('/content/neurovlm/experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn/materialized_primary_jsonl/mixed/val.jsonl'), 'test': PosixPath('/content/neurovlm/experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn/materialized_primary_jsonl/mixed/test.jsonl')}
Source counts by split:


,split,canonical_source,n
0,test,neurovault,202
1,test,nilearn,79
2,test,pubmed,3066
3,train,neurovault,1779
4,train,nilearn,639
5,train,pubmed,24526
6,val,neurovault,221
7,val,nilearn,79
8,val,pubmed,3066


Saved preprocessing/input stats: /content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline/full_atlas_free_cnn_20260528_053320/input_volume_stats_by_split_source.csv


,split,source,source_detail,n_maps,min,max,mean,std,fraction_nonzero_voxels,p95_sampled,p99_sampled,p995_sampled,empty_maps,nearly_empty_maps,targets_in_0_1,percentile_sample_voxels
0,test,all,all,3347,0.0,1.0,0.006348,0.050508,0.119151,0.000606,0.084290,0.203491,1,1,True,500000
1,test,neurovault,neurovault,202,0.0,1.0,0.055236,0.157876,0.200655,0.410400,0.799805,0.918457,0,0,True,200000
2,test,nilearn,nilearn,79,0.0,1.0,0.003484,0.051120,0.009474,0.000000,0.000000,0.182011,1,1,True,100000
3,test,pubmed,pubmed,3066,0.0,1.0,0.003201,0.030129,0.116608,0.000700,0.086548,0.201904,0,0,True,500000
14,train,all,all,26944,0.0,1.0,0.006634,0.052484,0.118451,0.000513,0.079468,0.198120,0,0,True,500000
15,train,neurovault,neurovault,1779,0.0,1.0,0.055784,0.160269,0.200792,0.527344,0.909180,1.000000,0,0,True,500000
16,train,nilearn,nilearn,639,0.0,1.0,0.004103,0.057055,0.010784,0.000000,0.000182,0.225464,0,0,True,500000
17,train,pubmed,pubmed,24526,0.0,1.0,0.003135,0.029842,0.115284,0.000513,0.072144,0.193359,0,0,True,500000
28,val,all,all,3366,0.0,1.0,0.006419,0.050855,0.117947,0.000468,0.070862,0.193359,0,0,True,500000
29,val,neurovault,neurovault,221,0.0,1.0,0.053132,0.153598,0.204651,0.383789,0.779297,0.932617,0,0,True,200000


## Stage 1: Train Atlas-Free CNN Autoencoder

Training objective:

```text
ALE volume -> CNN encoder -> latent -> CNN decoder -> reconstructed ALE volume
```

`AE_TRAINING_RECIPE="previous_good_compatible"` is the default for both `DATA_MODE="pubmed_only"` and `DATA_MODE="mixed"`: plain raw-output MSE, AdamW, `lr=3e-4`, AMP, gradient clipping, and checkpoint selection by lowest validation MSE.


In [6]:
training_file = ROOT / 'experiments/3dcnn/atlas_free_cnn/training/train_autoencoder.py'
if not training_file.exists():
    raise FileNotFoundError(
        f'Missing {training_file}. The Colab checkout is probably older than this notebook. '
        'Pull/clone the current repo version, or upload the current experiments/3dcnn/atlas_free_cnn folder.'
    )

try:
    import argparse
    import importlib.util
    import os
    import shutil
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader
    from atlas_free_cnn.training.model_wrappers import build_cnn_autoencoder
    from atlas_free_cnn.training.train_autoencoder import (
        VolumeCollator,
        train_from_config as train_autoencoder,
    )
    from atlas_free_cnn.training.datasets import UnifiedMapTextDataset
    from neurovlm.gnn.ale_cnn import ALE3DCNNAutoEncoder, count_parameters
except ModuleNotFoundError as exc:
    print('cwd:', Path.cwd())
    print('ROOT:', ROOT)
    print('first sys.path entries:', sys.path[:5])
    print('training file exists:', training_file.exists())
    raise exc

RUN_STAGE1_AUTOENCODER = globals().get('RUN_STAGE1_AUTOENCODER', True)
RETRAIN_STAGE1_IF_PRESENT = globals().get('RETRAIN_STAGE1_IF_PRESENT', False)
AUTO_BATCH_SIZE_PRECHECK = globals().get('AUTO_BATCH_SIZE_PRECHECK', True)
AE_PRIMARY_CHECKPOINT_NAME = globals().get('AE_PRIMARY_CHECKPOINT_NAME', 'best_cnn_autoencoder.pt')

OLD_GOOD_RECIPE = {
    'data_source_cache': 'legacy PubMed atlas-free ALE cache built from PubMed MNI coordinates, not HF packed mixed dataset',
    'mode': 'atlas_free',
    'resolution_mm': 4.0,
    'kernel_fwhm_mm': 9.0,
    'crop_to_brain': True,
    'normalize': 'max',
    'clamp': True,
    'cache_dtype': 'float16',
    'target_shape': TARGET_SHAPE,
    'model_class': 'ALE3DCNNAutoEncoder',
    'encoder_class': 'ALE3DCNNEncoder',
    'decoder_class': 'ALE3DCNNDecoder',
    'base_channels': 64,
    'num_blocks': 4,
    'latent_dim': 384,
    'dropout': 0.1,
    'norm': 'group',
    'pooling': 'max',
    'decoder_seed_shape': (3, 3, 3),
    'upsampling': 'ConvTranspose3d blocks, then trilinear interpolation to exact output shape if needed',
    'final_output_layer': 'Conv3d(base_channels, 1, kernel_size=3, padding=1)',
    'loss': 'plain MSE on raw decoder output vs continuous normalized ALE target',
    'activation_handling': 'no sigmoid in MSE loss; clamp output to [0, 1] only for plotting/evaluation display',
    'optimizer': 'AdamW',
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'batch_size': 256,
    'epochs': 200,
    'amp': True,
    'gradient_clipping': 1.0,
    'scheduler': 'none',
    'checkpoint_metric': 'lowest validation MSE loss',
    'checkpoint_name': 'best_cnn_autoencoder.pt',
    'plotting': 'middle axial slice in old notebook; new notebook also adds peak-centered and MIP diagnostics',
}

AE_MODE_CONFIGS = {
    'smoke': {
        'base_channels': 8,
        'num_blocks': 2,
        'latent_dim': 384,
        'epochs': 3,
        'batch_size_candidates': [8, 4, 2],
        'val_interval': 1,
        'max_train_batches': 20,
        'max_val_batches': 8,
        'train_metric_batches': 4,
        'val_metric_batches': 8,
    },
    'full': {
        'base_channels': 64,
        'num_blocks': 4,
        'latent_dim': 384,
        'epochs': 200,
        'batch_size_candidates': [1024, 768, 512, 384, 256, 192, 128, 96, 64],
        'val_interval': 5,
        'max_train_batches': None,
        'max_val_batches': None,
        'train_metric_batches': None,
        'val_metric_batches': None,
    },
}
AE_SELECTED_MODE = AE_MODE_CONFIGS[RUN_MODE]


def _legacy_cache_paths():
    cache_basename = f"atlas_free_ale_{int(OLD_GOOD_RECIPE['resolution_mm'])}mm_fwhm{str(OLD_GOOD_RECIPE['kernel_fwhm_mm']).replace('.', 'p')}_crop_{OLD_GOOD_RECIPE['cache_dtype']}.pt"
    if IN_COLAB:
        local_cache_dir = Path('/content/ale_caches_ale_3dcnn')
    else:
        local_cache_dir = ROOT / 'data/ale_caches'
    drive_cache_dir = DRIVE_ROOT / 'data_ale_3dcnn/ale_caches'
    local_cache_dir.mkdir(parents=True, exist_ok=True)
    drive_cache_dir.mkdir(parents=True, exist_ok=True)
    local_cache = local_cache_dir / cache_basename
    drive_cache = drive_cache_dir / cache_basename
    legacy_candidates = [
        drive_cache,
        drive_cache_dir / f"atlas_free_{int(OLD_GOOD_RECIPE['resolution_mm'])}mm_fwhm{int(OLD_GOOD_RECIPE['kernel_fwhm_mm'])}_crop_{OLD_GOOD_RECIPE['cache_dtype']}.pt",
        drive_cache_dir / f"atlas_free_{OLD_GOOD_RECIPE['resolution_mm']:g}mm_fwhm{OLD_GOOD_RECIPE['kernel_fwhm_mm']:g}_crop_{OLD_GOOD_RECIPE['cache_dtype']}.pt",
    ]
    for candidate in legacy_candidates:
        if candidate.exists() and not local_cache.exists():
            shutil.copy2(candidate, local_cache)
            print('Copied legacy ALE cache to local cache:', candidate, '->', local_cache)
            break
    return local_cache, drive_cache


def _load_train_ale_cnn_module():
    module_path = ROOT / 'experiments/3dcnn/train_ale_cnn.py'
    spec = importlib.util.spec_from_file_location('train_ale_cnn_legacy_recipe', module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def build_previous_good_dataset(stage1_dir):
    train_ale_cnn = _load_train_ale_cnn_module()
    legacy_cache, drive_cache = _legacy_cache_paths()
    dataset_args = argparse.Namespace(
        mode='atlas_free',
        kernel_fwhm_mm=OLD_GOOD_RECIPE['kernel_fwhm_mm'],
        resolution_mm=OLD_GOOD_RECIPE['resolution_mm'],
        crop_to_brain=OLD_GOOD_RECIPE['crop_to_brain'],
        normalize=OLD_GOOD_RECIPE['normalize'],
        no_clamp=not OLD_GOOD_RECIPE['clamp'],
        cache_dtype=OLD_GOOD_RECIPE['cache_dtype'],
        cache_file=str(legacy_cache),
        force_rebuild_cache=False,
        max_papers=None,
        run_dir=str(stage1_dir / 'legacy_pubmed_dataset_split'),
        seed=SEED,
        val_frac=0.1,
        test_frac=0.1,
    )
    ds, train_ds, val_ds, test_ds, cache_payload, preprocess_config = train_ale_cnn.build_dataset(dataset_args)
    if legacy_cache.exists() and not drive_cache.exists():
        shutil.copy2(legacy_cache, drive_cache)
        print('Copied newly built legacy ALE cache to Drive/local cache mirror:', drive_cache)
    print('Previous-good-compatible input shape:', ds.input_shape)
    print('Previous-good-compatible splits:', len(train_ds), len(val_ds), len(test_ds))
    return ds, train_ds, val_ds, test_ds, cache_payload, preprocess_config, legacy_cache


def make_previous_good_loader(split_ds, batch_size, shuffle):
    num_workers = int(OLD_GOOD_RECIPE.get('num_workers', 4) if DEVICE.type == 'cuda' else 0)
    return DataLoader(
        split_ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=DEVICE.type == 'cuda',
        persistent_workers=num_workers > 0,
    )


def previous_good_reconstruction_loss(logits, target, loss_name='mse'):
    target = target.float()
    if loss_name == 'mse':
        return F.mse_loss(logits, target)
    if loss_name == 'bce_logits':
        return F.binary_cross_entropy_with_logits(logits, target.clamp(0, 1))
    raise ValueError(loss_name)


def previous_good_display_reconstruction(logits, loss_name='mse'):
    return torch.sigmoid(logits) if loss_name == 'bce_logits' else logits.clamp(0, 1)


@torch.no_grad()
def eval_previous_good_autoencoder_loss(model, loader):
    model.eval()
    losses = []
    for batch in loader:
        x = batch['volume'].float().to(DEVICE, non_blocking=DEVICE.type == 'cuda')
        logits = model(x)
        losses.append(float(previous_good_reconstruction_loss(logits, x, 'mse').detach().cpu()))
    return float(np.mean(losses))


def train_previous_good_compatible_autoencoder(stage1_dir):
    ds, train_ds, val_ds, test_ds, cache_payload, preprocess_config, legacy_cache = build_previous_good_dataset(stage1_dir)
    if tuple(ds.input_shape) != tuple(TARGET_SHAPE):
        raise ValueError(f'Old-compatible dataset shape {ds.input_shape} does not match TARGET_SHAPE={TARGET_SHAPE}')
    ckpt_dir = stage1_dir / 'checkpoints'
    plot_dir = stage1_dir / 'plots'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    config = {
        'stage': 'cnn_autoencoder_pretraining',
        'recipe': 'previous_good_compatible',
        'mode': 'atlas_free',
        'input_shape': tuple(ds.input_shape),
        'target_shape': tuple(ds.input_shape),
        'latent_dim': OLD_GOOD_RECIPE['latent_dim'],
        'base_channels': OLD_GOOD_RECIPE['base_channels'],
        'num_blocks': OLD_GOOD_RECIPE['num_blocks'],
        'dropout': OLD_GOOD_RECIPE['dropout'],
        'norm': OLD_GOOD_RECIPE['norm'],
        'pooling': OLD_GOOD_RECIPE['pooling'],
        'loss': 'mse',
        'prediction_activation': 'none',
        'epochs': OLD_GOOD_RECIPE['epochs'],
        'batch_size': OLD_GOOD_RECIPE['batch_size'],
        'lr': OLD_GOOD_RECIPE['lr'],
        'weight_decay': OLD_GOOD_RECIPE['weight_decay'],
        'amp': bool(OLD_GOOD_RECIPE['amp']),
        'gradient_clipping': OLD_GOOD_RECIPE['gradient_clipping'],
        'checkpoint_metric': OLD_GOOD_RECIPE['checkpoint_metric'],
        'cache_file': str(legacy_cache),
        'preprocess_config': cache_payload['config'],
        'cache_metadata': cache_payload['metadata'],
    }
    (stage1_dir / 'autoencoder_config.json').write_text(json.dumps(config, indent=2, default=str))

    model = ALE3DCNNAutoEncoder(
        output_shape=ds.input_shape,
        base_channels=OLD_GOOD_RECIPE['base_channels'],
        num_blocks=OLD_GOOD_RECIPE['num_blocks'],
        latent_dim=OLD_GOOD_RECIPE['latent_dim'],
        dropout=OLD_GOOD_RECIPE['dropout'],
        norm=OLD_GOOD_RECIPE['norm'],
        pooling=OLD_GOOD_RECIPE['pooling'],
    ).to(DEVICE)
    print('Autoencoder params:', count_parameters(model))
    print('Decoder seed shape:', tuple(model.decoder.seed_shape))
    optimizer = torch.optim.AdamW(model.parameters(), lr=OLD_GOOD_RECIPE['lr'], weight_decay=OLD_GOOD_RECIPE['weight_decay'])
    use_amp = bool(OLD_GOOD_RECIPE['amp'] and DEVICE.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    train_loader = make_previous_good_loader(train_ds, OLD_GOOD_RECIPE['batch_size'], shuffle=True)
    val_loader = make_previous_good_loader(val_ds, OLD_GOOD_RECIPE['batch_size'], shuffle=False)
    history = {'train_loss': [], 'val_loss': [], 'epoch_time_sec': [], 'lr': []}
    best_loss = float('inf')
    best_state = None
    bad_val_checks = 0
    early_patience = 25

    for epoch in range(OLD_GOOD_RECIPE['epochs']):
        start = time.perf_counter()
        model.train()
        losses = []
        for batch in train_loader:
            x = batch['volume'].float().to(DEVICE, non_blocking=DEVICE.type == 'cuda')
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(x)
                loss = previous_good_reconstruction_loss(logits, x, 'mse')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), OLD_GOOD_RECIPE['gradient_clipping'])
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        val_loss = eval_previous_good_autoencoder_loss(model, val_loader)
        train_loss = float(np.mean(losses))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['epoch_time_sec'].append(float(time.perf_counter() - start))
        history['lr'].append(float(optimizer.param_groups[0]['lr']))
        print(f'Epoch {epoch:03d}/{OLD_GOOD_RECIPE["epochs"]} train={train_loss:.6f} val={val_loss:.6f}')
        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            raise FloatingPointError(f'NaN/Inf detected in previous-good-compatible AE training at epoch={epoch}: train={train_loss} val={val_loss}')
        state = {
            'encoder': model.encoder.state_dict(),
            'decoder': model.decoder.state_dict(),
            'model': model.state_dict(),
            'epoch': epoch,
            'val_loss': val_loss,
            'metrics': {'loss': val_loss, 'mse': val_loss},
            'history': history,
            'config': config,
            'target_shape': tuple(ds.input_shape),
        }
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = state
            torch.save(state, ckpt_dir / 'best_cnn_autoencoder.pt')
            torch.save(state, ckpt_dir / 'best_val_loss.pt')
            bad_val_checks = 0
        else:
            bad_val_checks += 1
            if bad_val_checks >= early_patience:
                print(f'Early stopping Stage 1 AE after {bad_val_checks} non-improving validation checks.')
                break

    torch.save(state, ckpt_dir / 'last_cnn_autoencoder.pt')
    torch.save(state, ckpt_dir / 'last.pt')
    (stage1_dir / 'autoencoder_training_history.json').write_text(json.dumps(history, indent=2))
    (stage1_dir / 'autoencoder_run_info.json').write_text(json.dumps({
        'run_dir': str(stage1_dir),
        'best_checkpoint': str(ckpt_dir / 'best_cnn_autoencoder.pt'),
        'last_checkpoint': str(ckpt_dir / 'last_cnn_autoencoder.pt'),
        'best_val_loss': best_loss,
        'recipe': 'previous_good_compatible',
    }, indent=2))
    return {
        'checkpoint_dir': str(ckpt_dir),
        'run_dir': str(stage1_dir),
        'best_checkpoint': str(ckpt_dir / 'best_cnn_autoencoder.pt'),
        'legacy_dataset': {'all': ds, 'train': train_ds, 'val': val_ds, 'test': test_ds},
        'cache_payload': cache_payload,
        'preprocess_config': preprocess_config,
        'config': config,
    }


def select_autoencoder_batch_size(cfg, candidates):
    if not AUTO_BATCH_SIZE_PRECHECK:
        return int(candidates[0])
    train_ds = UnifiedMapTextDataset(cfg['train_jsonl'])
    if len(train_ds) == 0:
        raise ValueError(f'No training rows for DATA_MODE={DATA_MODE}')
    for bs in candidates:
        try:
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            model_cfg = cfg['model']
            model = build_cnn_autoencoder(
                TARGET_SHAPE,
                latent_dim=int(model_cfg['latent_dim']),
                base_channels=int(model_cfg['base_channels']),
                num_blocks=int(model_cfg['num_blocks']),
                encoder_arch=str(model_cfg.get('encoder_arch', 'plain')),
            ).to(DEVICE)
            loader = DataLoader(train_ds, batch_size=min(int(bs), len(train_ds)), shuffle=False, collate_fn=VolumeCollator(TARGET_SHAPE), num_workers=0)
            batch = next(iter(loader))
            x = batch['volume'].to(DEVICE)
            model.train()
            pred = model(x)
            loss = F.mse_loss(pred, x)
            loss.backward()
            peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == 'cuda' else 0.0
            del model, loader, batch, x, pred, loss
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
            print(f'Selected Stage 1 batch_size={bs} peak_vram_mb={peak_mb:.0f}')
            return int(bs)
        except RuntimeError as exc:
            message = str(exc).lower()
            if 'out of memory' not in message and 'mps backend out of memory' not in message:
                raise
            print(f'Stage 1 batch_size={bs} OOM; trying smaller.')
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
    raise RuntimeError(f'No Stage 1 batch size fit from candidates={candidates}')


stage1_dir = OUT / f'stage1_autoencoder_{DATA_MODE}_{RUN_MODE}_{AE_TRAINING_RECIPE}'
ae_cfg = {
    'train_jsonl': str(SPLIT_JSONL['train']),
    'val_jsonl': str(SPLIT_JSONL['val']),
    'test_jsonl': str(SPLIT_JSONL.get('test', SPLIT_JSONL['val'])),
    'target_shape': list(TARGET_SHAPE),
    'output_dir': str(stage1_dir),
    'checkpoint_dir': str(stage1_dir / 'checkpoints'),
    'batch_size': OLD_GOOD_RECIPE['batch_size'],
    'epochs': OLD_GOOD_RECIPE['epochs'] if RUN_MODE == 'full' else int(AE_SELECTED_MODE['epochs']),
    'early_stopping': True,
    'early_stopping_metric': 'val_loss',
    'early_stopping_patience': 25,
    'early_stopping_min_delta': 0.0,
    'lr': OLD_GOOD_RECIPE['lr'],
    'weight_decay': OLD_GOOD_RECIPE['weight_decay'],
    'val_interval': int(AE_SELECTED_MODE['val_interval']),
    'include_voxel_auroc': True,
    'progress': True,
    'amp': True,
    'cudnn_benchmark': True,
    'num_workers': 4 if DEVICE.type == 'cuda' else 0,
    'pin_memory': DEVICE.type == 'cuda',
    'persistent_workers': DEVICE.type == 'cuda',
    'prefetch_factor': 4,
    'compute_train_metrics': RUN_MODE == 'smoke',
    'train_metric_batches': AE_SELECTED_MODE['train_metric_batches'],
    'val_metric_batches': AE_SELECTED_MODE['val_metric_batches'],
    'max_train_batbatches': AE_SELECTED_MODE['max_train_batches'],
    'max_val_batches': AE_SELECTED_MODE['max_val_batches'],
    'model': {
        'latent_dim': OLD_GOOD_RECIPE['latent_dim'] if AE_TRAINING_RECIPE == 'previous_good_compatible' else int(AE_SELECTED_MODE['latent_dim']),
        'base_channels': OLD_GOOD_RECIPE['base_channels'] if AE_TRAINING_RECIPE == 'previous_good_compatible' else int(AE_SELECTED_MODE['base_channels']),
        'num_blocks': OLD_GOOD_RECIPE['num_blocks'] if AE_TRAINING_RECIPE == 'previous_good_compatible' else int(AE_SELECTED_MODE['num_blocks']),
        'encoder_arch': 'plain',
        'dropout': OLD_GOOD_RECIPE['dropout'],
        'norm': OLD_GOOD_RECIPE['norm'],
        'pooling': OLD_GOOD_RECIPE['pooling'],
    },
    'loss': {'type': 'raw_mse'},
    'weighted_recon': {'type': 'mse', 'alpha': 0.0, 'gamma': 1.0},
    'prediction_activation': 'none',
    'run_mode': RUN_MODE,
    'data_mode': DATA_MODE,
    'ae_training_recipe': AE_TRAINING_RECIPE,
    'model_size': globals().get('MODEL_SIZE', 'base'),
    'data_mode': DATA_MODE,
    'source_sampling': globals().get('SOURCE_SAMPLING', 'temperature'),
    'source_sampling_alpha': globals().get('SOURCE_SAMPLING_ALPHA', 0.5),
    'preflight_batch_size': AUTO_BATCH_SIZE_PRECHECK,
    'preflight_vram_reserve_gb': 3.0,
    'batch_candidates': AE_SELECTED_MODE['batch_size_candidates'],
    'final_eval': True,
    'save_plots': True,
}

Path(ae_cfg['output_dir']).mkdir(parents=True, exist_ok=True)
Path(ae_cfg['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
(OUT / 'stage1_autoencoder_config.json').write_text(json.dumps(ae_cfg, indent=2, default=str))

recipe_comparison_static = pd.DataFrame([
    {
        'recipe': 'old notebook recipe from 2_ale_3dcnn_autoencoder_pretrain_colab.ipynb',
        'data source/cache': OLD_GOOD_RECIPE['data_source_cache'],
        'target shape': str(OLD_GOOD_RECIPE['target_shape']),
        'base_channels': OLD_GOOD_RECIPE['base_channels'],
        'num_blocks': OLD_GOOD_RECIPE['num_blocks'],
        'latent_dim': OLD_GOOD_RECIPE['latent_dim'],
        'loss': OLD_GOOD_RECIPE['loss'],
        'activation handling': OLD_GOOD_RECIPE['activation_handling'],
        'optimizer': OLD_GOOD_RECIPE['optimizer'],
        'lr': OLD_GOOD_RECIPE['lr'],
        'batch size': OLD_GOOD_RECIPE['batch_size'],
        'epochs': OLD_GOOD_RECIPE['epochs'],
        'AMP': OLD_GOOD_RECIPE['amp'],
        'checkpoint metric': OLD_GOOD_RECIPE['checkpoint_metric'],
    },
    {
        'recipe': 'new previous_good_compatible recipe',
        'data source/cache': 'same legacy PubMed ALE cache path/build_dataset path as old notebook',
        'target shape': str(TARGET_SHAPE),
        'base_channels': ae_cfg['model']['base_channels'],
        'num_blocks': ae_cfg['model']['num_blocks'],
        'latent_dim': ae_cfg['model']['latent_dim'],
        'loss': 'plain MSE on raw decoder output' if AE_TRAINING_RECIPE == 'previous_good_compatible' else 'not active',
        'activation handling': 'no sigmoid in loss; clamp only for eval/plots' if AE_TRAINING_RECIPE == 'previous_good_compatible' else 'not active',
        'optimizer': 'AdamW',
        'lr': ae_cfg['lr'],
        'batch size': ae_cfg['batch_size'],
        'epochs': ae_cfg['epochs'],
        'AMP': ae_cfg['amp'],
        'checkpoint metric': 'lowest validation MSE loss' if AE_TRAINING_RECIPE == 'previous_good_compatible' else 'not active',
    },
    {
        'recipe': 'quarantined sparse-loss ablation (not runnable from this pipeline)',
        'data source/cache': 'HF packed atlas_free_cnn primary JSONL rows',
        'target shape': str(TARGET_SHAPE),
        'base_channels': 64,
        'num_blocks': 4,
        'latent_dim': 384,
        'loss': 'deleted old sparse generation loss ablation',
        'activation handling': 'quarantined old setup: sigmoid before sparse loss; not a Stage 1 default',
        'optimizer': 'AdamW',
        'lr': 3e-4,
        'batch size': 'auto-preflight candidates up to 512',
        'epochs': 200,
        'AMP': True,
        'checkpoint metric': 'best_val_loss / sparse metrics',
    },
])
recipe_comparison_static.to_csv(OUT / 'autoencoder_recipe_comparison_static.csv', index=False)
print('Stage 1 autoencoder config:')
print(json.dumps({
    'run_mode': RUN_MODE,
    'data_mode': DATA_MODE,
    'ae_training_recipe': AE_TRAINING_RECIPE,
    'base_channels': ae_cfg['model']['base_channels'],
    'num_blocks': ae_cfg['model']['num_blocks'],
    'latent_dim': ae_cfg['model']['latent_dim'],
    'epochs': ae_cfg['epochs'],
    'batch_size': ae_cfg['batch_size'],
    'checkpoint_dir': ae_cfg['checkpoint_dir'],
}, indent=2))
display(recipe_comparison_static)

AE_CKPT = Path(ae_cfg['checkpoint_dir']) / AE_PRIMARY_CHECKPOINT_NAME
LEGACY_PUBMED_DATASETS = None
if AE_TRAINING_RECIPE == 'previous_good_compatible' and DATA_MODE == 'pubmed_only':
    if RUN_STAGE1_AUTOENCODER and (RETRAIN_STAGE1_IF_PRESENT or not AE_CKPT.exists()):
        print('Training Stage 1 autoencoder with previous_good_compatible PubMed-only plain-MSE recipe.')
        ae_result = train_previous_good_compatible_autoencoder(stage1_dir)
        LEGACY_PUBMED_DATASETS = ae_result['legacy_dataset']
        LEGACY_CACHE_PAYLOAD = ae_result['cache_payload']
        LEGACY_PREPROCESS_CONFIG = ae_result['preprocess_config']
        AE_CKPT = Path(ae_result['best_checkpoint'])
    elif AE_CKPT.exists():
        print('Using existing previous_good_compatible PubMed-only Stage 1 checkpoint:', AE_CKPT)
        ds, train_ds, val_ds, test_ds, cache_payload, preprocess_config, legacy_cache = build_previous_good_dataset(stage1_dir)
        LEGACY_PUBMED_DATASETS = {'all': ds, 'train': train_ds, 'val': val_ds, 'test': test_ds}
        LEGACY_CACHE_PAYLOAD = cache_payload
        LEGACY_PREPROCESS_CONFIG = preprocess_config
    else:
        raise FileNotFoundError('RUN_STAGE1_AUTOENCODER is False and no previous_good_compatible checkpoint exists: ' + str(AE_CKPT))
elif AE_TRAINING_RECIPE == 'previous_good_compatible' and DATA_MODE == 'mixed':
    if RUN_STAGE1_AUTOENCODER and (RETRAIN_STAGE1_IF_PRESENT or not AE_CKPT.exists()):
        print('Training Stage 1 autoencoder with previous_good_compatible mixed-source raw-MSE recipe.')
        ae_result = train_autoencoder(ae_cfg)
        AE_CKPT = Path(ae_result['best_checkpoint'])
    elif AE_CKPT.exists():
        print('Using existing previous_good_compatible mixed-source Stage 1 checkpoint:', AE_CKPT)
    else:
        raise FileNotFoundError('RUN_STAGE1_AUTOENCODER is False and no mixed-source Stage 1 checkpoint exists: ' + str(AE_CKPT))
else:
    raise RuntimeError('Only previous_good_compatible Stage 1 is available in the clean 3D CNN pipeline.')

print('Stage 1 primary checkpoint:', AE_CKPT)
AE_CKPT

Stage 1 autoencoder config:
{
  "run_mode": "full",
  "data_mode": "mixed",
  "ae_training_recipe": "previous_good_compatible",
  "base_channels": 64,
  "num_blocks": 4,
  "latent_dim": 384,
  "epochs": 200,
  "batch_size": 256,
  "checkpoint_dir": "/content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline/full_atlas_free_cnn_20260528_053320/stage1_autoencoder_mixed_full_previous_good_compatible/checkpoints"
}


,recipe,data source/cache,target shape,base_channels,num_blocks,latent_dim,loss,activation handling,optimizer,lr,batch size,epochs,AMP,checkpoint metric
0,old notebook recipe from 2_ale_3dcnn_autoencod...,legacy PubMed atlas-free ALE cache built from ...,"(36, 45, 38)",64,4,384,plain MSE on raw decoder output vs continuous ...,"no sigmoid in MSE loss; clamp output to [0, 1]...",AdamW,0.0003,256,200,True,lowest validation MSE loss
1,new previous_good_compatible recipe,same legacy PubMed ALE cache path/build_datase...,"(36, 45, 38)",64,4,384,plain MSE on raw decoder output,no sigmoid in loss; clamp only for eval/plots,AdamW,0.0003,256,200,True,lowest validation MSE loss
2,quarantined sparse-loss ablation (not runnable...,HF packed atlas_free_cnn primary JSONL rows,"(36, 45, 38)",64,4,384,deleted old sparse generation loss ablation,quarantined old setup: sigmoid before sparse l...,AdamW,0.0003,auto-preflight candidates up to 512,200,True,best_val_loss / sparse metrics


Training Stage 1 autoencoder with previous_good_compatible mixed-source raw-MSE recipe.
{'data_mode': 'mixed', 'source_counts_by_split': {'train': {'pubmed': 24526, 'nilearn': 639, 'neurovault': 1779}, 'val': {'pubmed': 3066, 'nilearn': 79, 'neurovault': 221}, 'test': {'pubmed': 3066, 'nilearn': 79, 'neurovault': 202}}}


/content/neurovlm/experiments/3dcnn/atlas_free_cnn/training/train_autoencoder.py:190: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(cfg.get("amp", True))):
/content/neurovlm/experiments/3dcnn/atlas_free_cnn/training/train_autoencoder.py:404: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=bool(use_amp and device.type == "cuda"))


epoch 1 train:   0%|          | 0/71 [00:00<?, ?batch/s]

...

{'epoch': 159, 'train_loss': 0.000714869081849416, 'train_epoch_time_sec': 35.98052358627319, 'train_source_counts': {'neurovault': 5040, 'pubmed': 18846, 'nilearn': 3058}, 'train_peak_vram_gb': 69.51178312301636, 'val_mse': 0.0009421133727300912, 'val_mae': 0.005251526832580566, 'val_spatial_corr': 0.7644294251998266, 'val_top1_dice': 0.5699221028221978, 'val_top5_dice': 0.518379827340444, 'val_top10_dice': 0.45905615554915535, 'val_top1_overlap': 0.5699221028221978, 'val_top5_overlap': 0.518379827340444, 'val_top10_overlap': 0.45905615554915535, 'val_target_nonzero_fraction': 0.11957957098881404, 'val_pred_nonzero_fraction': 0.5214909182654487, 'val_pred_mean': 0.00772015621057815, 'val_pred_max': 1.0, 'val_voxel_auroc': nan, 'val_loss': 0.0009500052191368821, 'val_epoch_time_sec': 17.798162698745728, 'val_source_counts': {'pubmed': 3066, 'nilearn': 79, 'neurovault': 221}, 'val_peak_vram_gb': 69.51168394088745}


epoch 160 train:   0%|          | 0/71 [00:00<?, ?batch/s]

epoch 160 val:   0%|          | 0/9 [00:00<?, ?batch/s]

{'early_stopping': True, 'epoch': 160, 'metric': 'val_loss', 'best_value': 0.0009419448841880593, 'current_value': 0.0009656353504396975, 'bad_val_checks': 25}
Stage 1 primary checkpoint: /content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline/full_atlas_free_cnn_20260528_053320/stage1_autoencoder_mixed_full_previous_good_compatible/checkpoints/best_cnn_autoencoder.pt


PosixPath('/content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_full_pipeline/full_atlas_free_cnn_20260528_053320/stage1_autoencoder_mixed_full_previous_good_compatible/checkpoints/best_cnn_autoencoder.pt')

## Stage 1 Eval: Reconstruction Metrics, Checkpoint Comparison, And Qualitative Examples

This cell loads every requested checkpoint variant, prints checkpoint config and layer shapes before evaluation, reports metrics by source, and saves random middle-slice, peak-centered, and max-intensity projection plots. It also evaluates the previous known-good PubMed-only checkpoint when it is available at `runs_ale_3dcnn_autoencoder_pretrain/autoencoder_atlas_free_20260510_194641`.


In [7]:

import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
from torch.utils.data import DataLoader

from atlas_free_cnn.evaluation.generation_metrics import generation_metrics
from atlas_free_cnn.training.datasets import UnifiedMapTextDataset
from atlas_free_cnn.training.train_autoencoder import VolumeCollator
from atlas_free_cnn.training.model_wrappers import build_cnn_autoencoder
from neurovlm.gnn.ale_cnn import count_parameters


def mae(pred, target):
    return float((pred - target).abs().mean().item())


def _checkpoint_config(payload):
    return payload.get('config', {}) if isinstance(payload, dict) else {}


def _model_cfg_from_checkpoint(payload, fallback_cfg=None):
    cfg = _checkpoint_config(payload)
    fallback_cfg = fallback_cfg or {}
    nested = cfg.get('model', {}) if isinstance(cfg.get('model', {}), dict) else {}
    target_shape = tuple(cfg.get('target_shape') or cfg.get('input_shape') or payload.get('target_shape', TARGET_SHAPE))
    return {
        'target_shape': target_shape,
        'latent_dim': int(nested.get('latent_dim', cfg.get('latent_dim', fallback_cfg.get('latent_dim', 384)))),
        'base_channels': int(nested.get('base_channels', cfg.get('base_channels', fallback_cfg.get('base_channels', 48)))),
        'num_blocks': int(nested.get('num_blocks', cfg.get('num_blocks', fallback_cfg.get('num_blocks', 4)))),
        'encoder_arch': str(nested.get('encoder_arch', cfg.get('encoder_arch', fallback_cfg.get('encoder_arch', 'plain')))),
        'blocks_per_stage': int(nested.get('blocks_per_stage', cfg.get('blocks_per_stage', fallback_cfg.get('blocks_per_stage', 2)))),
        'use_dilation': bool(nested.get('use_dilation', cfg.get('use_dilation', fallback_cfg.get('use_dilation', False)))),
        'multi_scale': bool(nested.get('multi_scale', cfg.get('multi_scale', fallback_cfg.get('multi_scale', False)))),
        'global_context': str(nested.get('global_context', cfg.get('global_context', fallback_cfg.get('global_context', 'none')))),
    }


def _state_dict_from_payload(payload):
    state = payload.get('model') or payload.get('autoencoder') or payload.get('state_dict')
    if state is None:
        raise KeyError("Checkpoint must contain 'model', 'autoencoder', or 'state_dict'")
    if any(str(k).startswith('_orig_mod.') for k in state):
        state = {str(k).replace('_orig_mod.', '', 1): v for k, v in state.items()}
    return state


def _prediction_activation_from_checkpoint(payload):
    cfg = _checkpoint_config(payload)
    if 'prediction_activation' in cfg:
        return str(cfg['prediction_activation'])
    if cfg.get('loss') == 'mse':
        return 'clamp'
    if cfg.get('loss') == 'bce_logits':
        return 'sigmoid'
    return 'none'


def activate_reconstruction(logits, payload):
    activation = _prediction_activation_from_checkpoint(payload)
    if activation == 'sigmoid':
        return torch.sigmoid(logits)
    if activation == 'softplus':
        return torch.nn.functional.softplus(logits)
    if activation == 'clamp':
        return logits.clamp(0, 1)
    if activation in {'none', 'linear'}:
        return logits.clamp(0, 1)
    raise ValueError(f'Unknown prediction activation: {activation}')


def load_autoencoder_for_eval(checkpoint_path, *, fallback_model_cfg=None, label='checkpoint'):
    checkpoint_path = Path(checkpoint_path)
    payload = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model_cfg = _model_cfg_from_checkpoint(payload, fallback_model_cfg)
    model = build_cnn_autoencoder(
        model_cfg['target_shape'],
        latent_dim=model_cfg['latent_dim'],
        base_channels=model_cfg['base_channels'],
        num_blocks=model_cfg['num_blocks'],
        encoder_arch=model_cfg['encoder_arch'],
        blocks_per_stage=model_cfg['blocks_per_stage'],
        use_dilation=model_cfg['use_dilation'],
        multi_scale=model_cfg['multi_scale'],
        global_context=model_cfg['global_context'],
    ).to(DEVICE)
    model.load_state_dict(_state_dict_from_payload(payload), strict=True)
    model.eval()
    print_autoencoder_report(label, checkpoint_path, model, payload, model_cfg)
    return model, payload, model_cfg


def _conv_channels(module):
    channels = []
    for name, layer in module.named_modules():
        if isinstance(layer, (nn.Conv3d, nn.ConvTranspose3d)):
            channels.append({'layer': name, 'type': layer.__class__.__name__, 'in_channels': int(layer.in_channels), 'out_channels': int(layer.out_channels), 'kernel_size': tuple(layer.kernel_size), 'stride': tuple(layer.stride)})
    return channels


def print_autoencoder_report(label, checkpoint_path, model, payload, model_cfg):
    cfg = _checkpoint_config(payload)
    activation = _prediction_activation_from_checkpoint(payload)
    with torch.no_grad():
        dummy = torch.zeros(1, 1, *model_cfg['target_shape'], device=DEVICE)
        logits = model(dummy)
        latent = model.encoder(dummy)
        recon = activate_reconstruction(logits, payload)
    report = {
        'label': label,
        'checkpoint': str(checkpoint_path),
        'checkpoint_epoch': payload.get('epoch'),
        'checkpoint_metrics': payload.get('metrics', {}),
        'checkpoint_config': cfg,
        'model_cfg': model_cfg,
        'parameter_count': int(count_parameters(model)),
        'encoder_conv_channels': _conv_channels(model.encoder),
        'decoder_conv_channels': _conv_channels(model.decoder),
        'latent_shape': tuple(latent.shape),
        'latent_dim': int(model_cfg['latent_dim']),
        'decoder_seed_shape': tuple(getattr(model.decoder, 'seed_shape', ())),
        'decoder_output_shape': tuple(logits.shape),
        'eval_reconstruction_shape': tuple(recon.shape),
        'decoder_raw_output_interpretation': 'raw scores/logits; old MSE recipe treats them as linear values during loss',
        'loss_prediction_activation': activation,
        'scaling_check': 'targets are expected in [0, 1]; old-compatible recipe uses raw-output MSE and clamps only for evaluation/plotting',
    }
    print('\nAUTOENCODER CHECKPOINT REPORT')
    print(json.dumps(report, indent=2, default=str))
    return report


def _metrics_for_pair(recon_i, true_i):
    m = generation_metrics(recon_i, true_i, include_voxel_auroc=True)
    m['reconstruction_mse'] = float((recon_i - true_i).pow(2).mean().item())
    m['mae'] = mae(recon_i, true_i)
    return m


def evaluate_reconstruction_split(model, payload, jsonl_path, split, batch_size=8, checkpoint_label='checkpoint'):
    ds = UnifiedMapTextDataset(jsonl_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=VolumeCollator(TARGET_SHAPE))
    rows_out = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            x = batch['volume'].to(DEVICE)
            recon = activate_reconstruction(model(x), payload)
            for i, map_id in enumerate(batch['map_id']):
                meta = batch['metadata'][i]
                true_i = x[i:i+1].detach().cpu()
                recon_i = recon[i:i+1].detach().cpu()
                m = _metrics_for_pair(recon_i, true_i)
                m.update({'checkpoint_label': checkpoint_label, 'map_id': map_id, 'split': split, 'source': meta.get('canonical_source', canonical_source(meta.get('source'))), 'source_detail': meta.get('source_detail', source_detail(meta)), 'target_min': float(true_i.min().item()), 'target_max': float(true_i.max().item()), 'target_fraction_nonzero': float((true_i != 0).float().mean().item())})
                rows_out.append(m)
    return pd.DataFrame(rows_out)


def evaluate_reconstruction_dataset(model, payload, dataset, split, batch_size=64, checkpoint_label='checkpoint', source='pubmed'):
    loader = make_previous_good_loader(dataset, batch_size, shuffle=False) if 'make_previous_good_loader' in globals() else DataLoader(dataset, batch_size=batch_size, shuffle=False)
    rows_out = []
    model.eval()
    positions = getattr(dataset, '_positions', None)
    parent_pmids = getattr(getattr(dataset, '_parent', None), 'pmids', None)
    with torch.no_grad():
        seen = 0
        for batch in loader:
            x = batch['volume'].float().to(DEVICE, non_blocking=DEVICE.type == 'cuda')
            recon = activate_reconstruction(model(x), payload)
            bsz = x.shape[0]
            for i in range(bsz):
                global_i = seen + i
                if parent_pmids is not None and positions is not None:
                    map_id = str(parent_pmids[positions[global_i]])
                else:
                    map_id = str(batch.get('paper_idx', torch.arange(seen, seen + bsz))[i].item())
                true_i = x[i:i+1].detach().cpu()
                recon_i = recon[i:i+1].detach().cpu()
                m = _metrics_for_pair(recon_i, true_i)
                m.update({'checkpoint_label': checkpoint_label, 'map_id': map_id, 'split': split, 'source': source, 'source_detail': source, 'target_min': float(true_i.min().item()), 'target_max': float(true_i.max().item()), 'target_fraction_nonzero': float((true_i != 0).float().mean().item())})
                rows_out.append(m)
            seen += bsz
    return pd.DataFrame(rows_out)


def _array3d(vol):
    return vol.squeeze().detach().cpu().numpy()


def _middle_center(arr):
    return tuple(int(s // 2) for s in arr.shape)


def _peak_center(arr):
    return tuple(int(v) for v in np.unravel_index(int(np.argmax(arr)), arr.shape))


def _slices_at(vol, center):
    arr = _array3d(vol)
    d, h, w = center
    return [arr[d, :, :], arr[:, h, :], arr[:, :, w]]


def _mip_slices(vol):
    arr = _array3d(vol)
    return [arr.max(axis=0), arr.max(axis=1), arr.max(axis=2)]


def _panels_for_view(true, recon, view):
    diff = (recon - true).abs()
    if view == 'middle':
        center = _middle_center(_array3d(true)); panels = _slices_at(true, center) + _slices_at(recon, center) + _slices_at(diff, center); title_prefix = f'middle {center}'
    elif view == 'peak':
        center = _peak_center(_array3d(true)); panels = _slices_at(true, center) + _slices_at(recon, center) + _slices_at(diff, center); title_prefix = f'peak {center}'
    elif view == 'mip':
        panels = _mip_slices(true) + _mip_slices(recon) + _mip_slices(diff); title_prefix = 'max intensity projection'
    else:
        raise ValueError(view)
    return panels, title_prefix


def _draw_reconstruction_grid(examples, out_png, out_dir, *, split, view, checkpoint_label):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(len(examples), 9, figsize=(18, 2.25 * len(examples)))
    if len(examples) == 1:
        axes = np.expand_dims(axes, 0)
    for r, (map_id, src, true, recon) in enumerate(examples):
        panels, title_prefix = _panels_for_view(true, recon, view)
        metric = generation_metrics(recon.unsqueeze(0), true.unsqueeze(0), include_voxel_auroc=False)
        vmax = max(float(true.max()), float(recon.max()), 1e-6)
        titles = ['orig axial','orig coronal','orig sagittal','recon axial','recon coronal','recon sagittal','diff axial','diff coronal','diff sagittal']
        for c, panel in enumerate(panels):
            axes[r, c].imshow(panel.T, origin='lower', cmap='magma' if c < 6 else 'viridis', vmin=0, vmax=vmax if c < 6 else None)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].set_title(titles[c], fontsize=8)
        axes[r, 0].set_ylabel(f"{checkpoint_label}\n{split}/{src}\n{map_id}\n{title_prefix}\nr={metric['spatial_corr']:.2f} d5={metric['top5_dice']:.2f}", fontsize=7)
        fig.savefig(out_dir / f"{split}_{view}_{src}_{map_id}.png", dpi=160, bbox_inches='tight')
    fig.tight_layout(); fig.savefig(out_png, dpi=180, bbox_inches='tight'); plt.close(fig)
    return str(out_png), str(out_dir)


def plot_reconstruction_examples(model, payload, jsonl_path, split, out_png, out_dir, *, n=6, view='middle', source_filter=None, checkpoint_label='checkpoint'):
    ds = UnifiedMapTextDataset(jsonl_path)
    candidates = [(idx, row) for idx, row in enumerate(ds.rows) if source_filter is None or row.get('canonical_source', canonical_source(row.get('source'))) == source_filter]
    if not candidates:
        print(f'No examples for split={split} source={source_filter}; skipping {view} plot.'); return None
    chosen = random.sample(candidates, k=min(n, len(candidates)))
    examples = []
    model.eval()
    for idx, row in chosen:
        item = ds[idx]
        x = item['volume'].unsqueeze(0).to(DEVICE)
        with torch.no_grad(): recon = activate_reconstruction(model(x), payload).cpu()[0]
        true = item['volume'].cpu()
        src = item['metadata'].get('canonical_source', canonical_source(item['metadata'].get('source')))
        examples.append((item['map_id'], src, true, recon))
    return _draw_reconstruction_grid(examples, out_png, out_dir, split=split, view=view, checkpoint_label=checkpoint_label)


def plot_reconstruction_dataset_examples(model, payload, dataset, split, out_png, out_dir, *, n=6, view='middle', checkpoint_label='checkpoint'):
    idxs = random.sample(range(len(dataset)), k=min(n, len(dataset)))
    positions = getattr(dataset, '_positions', None)
    parent_pmids = getattr(getattr(dataset, '_parent', None), 'pmids', None)
    examples = []
    model.eval()
    for idx in idxs:
        item = dataset[idx]
        x = item['volume'].unsqueeze(0).float().to(DEVICE)
        with torch.no_grad(): recon = activate_reconstruction(model(x), payload).cpu()[0]
        true = item['volume'].cpu()
        map_id = str(parent_pmids[positions[idx]]) if parent_pmids is not None and positions is not None else str(idx)
        examples.append((map_id, 'pubmed', true, recon))
    return _draw_reconstruction_grid(examples, out_png, out_dir, split=split, view=view, checkpoint_label=checkpoint_label)


def checkpoint_candidates_from_dir(ckpt_dir):
    ckpt_dir = Path(ckpt_dir)
    names = ['best_cnn_autoencoder.pt', 'best_val_loss.pt', 'best_spatial_corr.pt', 'best_top1_dice.pt', 'best_top5_dice.pt', 'best_generation_top5_dice.pt', 'last.pt', 'last_cnn_autoencoder.pt']
    aliases = {'best_spatial_corr.pt': 'best_generation_spatial_correlation.pt', 'best_top5_dice.pt': 'best_generation_top5_dice.pt', 'last.pt': 'last_cnn_autoencoder.pt'}
    out = {}
    for name in names:
        path = ckpt_dir / name
        if not path.exists() and name in aliases:
            path = ckpt_dir / aliases[name]
        if path.exists():
            out[name.replace('.pt', '')] = path
    return out


def resolve_previous_good_checkpoint():
    candidates = [
        ROOT / 'runs_ale_3dcnn_autoencoder_pretrain/autoencoder_atlas_free_20260510_194641/checkpoints/best_cnn_autoencoder.pt',
        DRIVE_ROOT / 'runs_ale_3dcnn_autoencoder_pretrain/autoencoder_atlas_free_20260510_194641/checkpoints/best_cnn_autoencoder.pt',
        Path('/content/drive/MyDrive/neurovlm/runs_ale_3dcnn_autoencoder_pretrain/autoencoder_atlas_free_20260510_194641/checkpoints/best_cnn_autoencoder.pt'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    print('Previous good PubMed-only checkpoint not found. Checked:')
    for candidate in candidates:
        print('  ', candidate)
    return None


RUN_STAGE1_RECON_EVAL = True
RECON_EVAL_BATCH_SIZE = max(1, min(int(ae_cfg.get('batch_size', 8)), 64))
PREVIOUS_GOOD_AE_CKPT = globals().get('PREVIOUS_GOOD_AE_CKPT', None) or resolve_previous_good_checkpoint()
SMOKE_TEST_AE_CKPT = globals().get('SMOKE_TEST_AE_CKPT', None)

recon_results = {}
reconstruction_plot_paths = {}
checkpoint_eval_rows = []
checkpoint_source_summary = pd.DataFrame()
recipe_comparison = recipe_comparison_static.copy() if 'recipe_comparison_static' in globals() else pd.DataFrame()

if RUN_STAGE1_RECON_EVAL:
    current_ckpts = checkpoint_candidates_from_dir(Path(ae_cfg['checkpoint_dir']))
    if not current_ckpts and Path(AE_CKPT).exists():
        current_ckpts = {'selected': Path(AE_CKPT)}

    eval_jobs = []
    if AE_TRAINING_RECIPE == 'previous_good_compatible' and DATA_MODE == 'pubmed_only':
        if LEGACY_PUBMED_DATASETS is None:
            raise RuntimeError('LEGACY_PUBMED_DATASETS is missing; rerun Stage 1 with DATA_MODE="pubmed_only" and AE_TRAINING_RECIPE="previous_good_compatible".')
        for label, ckpt_path in current_ckpts.items():
            eval_jobs.append((f'new_previous_good_compatible_{label}', ckpt_path, {'legacy_pubmed_test': LEGACY_PUBMED_DATASETS['test']}))
        if PREVIOUS_GOOD_AE_CKPT:
            eval_jobs.append(('old_checkpoint_best_cnn_autoencoder', Path(PREVIOUS_GOOD_AE_CKPT), {'legacy_pubmed_test': LEGACY_PUBMED_DATASETS['test']}))
    else:
        current_label_prefix = 'new_previous_good_compatible' if AE_TRAINING_RECIPE == 'previous_good_compatible' else f'current_{RUN_MODE}_{DATA_MODE}'
        current_eval_splits = {'pubmed_only_test': ALL_SPLIT_JSONL['pubmed_only']['test']}
        if DATA_MODE == 'mixed':
            current_eval_splits['mixed_test'] = ALL_SPLIT_JSONL['mixed']['test']
        for label, ckpt_path in current_ckpts.items():
            eval_jobs.append((f'{current_label_prefix}_{label}', ckpt_path, current_eval_splits))
        if PREVIOUS_GOOD_AE_CKPT:
            eval_jobs.append(('previous_good_pubmed_only_best_cnn_autoencoder', Path(PREVIOUS_GOOD_AE_CKPT), {'pubmed_only_test': ALL_SPLIT_JSONL['pubmed_only']['test']}))
        if SMOKE_TEST_AE_CKPT:
            eval_jobs.append(('new_tiny_smoke_test_checkpoint', Path(SMOKE_TEST_AE_CKPT), {'mixed_test': ALL_SPLIT_JSONL['mixed']['test']}))

    fallback_model_cfg = ae_cfg['model']
    for checkpoint_label, ckpt_path, split_map in eval_jobs:
        if not Path(ckpt_path).exists():
            print('Skipping missing checkpoint:', checkpoint_label, ckpt_path); continue
        model, payload, model_cfg = load_autoencoder_for_eval(ckpt_path, fallback_model_cfg=fallback_model_cfg, label=checkpoint_label)
        for eval_split, split_obj in split_map.items():
            print('Evaluating reconstruction:', checkpoint_label, eval_split)
            if isinstance(split_obj, (str, Path)):
                df = evaluate_reconstruction_split(model, payload, split_obj, eval_split, batch_size=RECON_EVAL_BATCH_SIZE, checkpoint_label=checkpoint_label)
            else:
                df = evaluate_reconstruction_dataset(model, payload, split_obj, eval_split, batch_size=RECON_EVAL_BATCH_SIZE, checkpoint_label=checkpoint_label, source='pubmed')
            csv_path = OUT / f'stage1_reconstruction_metrics_{checkpoint_label}_{eval_split}.csv'
            df.to_csv(csv_path, index=False)
            recon_results[(checkpoint_label, eval_split)] = df
            checkpoint_eval_rows.append(df)

        should_plot = checkpoint_label.startswith('new_previous_good_compatible_best_cnn_autoencoder') or checkpoint_label == 'old_checkpoint_best_cnn_autoencoder' or (checkpoint_label.startswith(f'current_{RUN_MODE}_{DATA_MODE}') and ckpt_path == Path(AE_CKPT))
        if should_plot:
            if AE_TRAINING_RECIPE == 'previous_good_compatible' and DATA_MODE == 'pubmed_only':
                plot_split_obj = LEGACY_PUBMED_DATASETS['test']; plot_split_name = 'legacy_pubmed_test'
                for view in ['middle', 'peak', 'mip']:
                    reconstruction_plot_paths[f'{checkpoint_label}_{plot_split_name}_{view}'] = plot_reconstruction_dataset_examples(
                        model, payload, plot_split_obj, plot_split_name, OUT / f'{checkpoint_label}_{plot_split_name}_{view}_reconstruction.png', OUT / f'{checkpoint_label}_{plot_split_name}_{view}_reconstruction', view=view, checkpoint_label=checkpoint_label
                    )
            else:
                for eval_split, jsonl_path in {'val': SPLIT_JSONL['val'], 'test': SPLIT_JSONL['test']}.items():
                    for view in ['middle', 'peak', 'mip']:
                        reconstruction_plot_paths[f'{checkpoint_label}_{eval_split}_{view}'] = plot_reconstruction_examples(
                            model, payload, jsonl_path, eval_split, OUT / f'{checkpoint_label}_{eval_split}_{view}_reconstruction.png', OUT / f'{checkpoint_label}_{eval_split}_{view}_reconstruction', view=view, checkpoint_label=checkpoint_label
                        )
                    for src in ['pubmed', 'neurovault', 'nilearn']:
                        reconstruction_plot_paths[f'{checkpoint_label}_{eval_split}_{src}_peak'] = plot_reconstruction_examples(
                            model, payload, jsonl_path, eval_split, OUT / f'{checkpoint_label}_{eval_split}_{src}_peak_reconstruction.png', OUT / f'{checkpoint_label}_{eval_split}_{src}_peak_reconstruction', view='peak', source_filter=src, checkpoint_label=checkpoint_label
                        )

    if checkpoint_eval_rows:
        all_recon_metrics = pd.concat(checkpoint_eval_rows, ignore_index=True)
        all_recon_metrics.to_csv(OUT / 'stage1_all_checkpoint_reconstruction_metrics.csv', index=False)
        metric_cols = ['reconstruction_mse', 'mse', 'mae', 'spatial_corr', 'top1_dice', 'top5_dice', 'top10_dice', 'voxel_auroc', 'target_min', 'target_max', 'target_fraction_nonzero']
        checkpoint_source_summary = all_recon_metrics.groupby(['checkpoint_label', 'split', 'source'])[metric_cols].mean(numeric_only=True).reset_index()
        checkpoint_source_summary.to_csv(OUT / 'stage1_reconstruction_summary_by_checkpoint_source.csv', index=False)

        def _first_metric(label_contains, metric):
            sub = checkpoint_source_summary[checkpoint_source_summary['checkpoint_label'].astype(str).str.contains(label_contains, regex=False)]
            if len(sub) == 0 or metric not in sub:
                return np.nan
            return float(pd.to_numeric(sub[metric], errors='coerce').mean())
        def _first_plot(label_contains):
            for key, paths in reconstruction_plot_paths.items():
                if label_contains in key and paths is not None:
                    return paths[0]
            return ''
        if len(recipe_comparison):
            metric_map = {
                'old notebook recipe from 2_ale_3dcnn_autoencoder_pretrain_colab.ipynb': 'old_checkpoint_best_cnn_autoencoder',
                'new previous_good_compatible recipe': 'new_previous_good_compatible',
                'quarantined sparse-loss ablation (not runnable from this pipeline)': 'current_full_pubmed_only',
            }
            for col in ['reconstruction MSE', 'spatial correlation', 'top1 Dice', 'top5 Dice', 'top10 Dice', 'qualitative plot path']:
                recipe_comparison[col] = ''
            for idx, row in recipe_comparison.iterrows():
                label = metric_map.get(row['recipe'], '')
                if label:
                    recipe_comparison.loc[idx, 'reconstruction MSE'] = _first_metric(label, 'reconstruction_mse')
                    recipe_comparison.loc[idx, 'spatial correlation'] = _first_metric(label, 'spatial_corr')
                    recipe_comparison.loc[idx, 'top1 Dice'] = _first_metric(label, 'top1_dice')
                    recipe_comparison.loc[idx, 'top5 Dice'] = _first_metric(label, 'top5_dice')
                    recipe_comparison.loc[idx, 'top10 Dice'] = _first_metric(label, 'top10_dice')
                    recipe_comparison.loc[idx, 'qualitative plot path'] = _first_plot(label)
            recipe_comparison.to_csv(OUT / 'autoencoder_recipe_comparison_with_metrics.csv', index=False)
            print('Autoencoder recipe comparison:')
            display(recipe_comparison)
        print('Saved reconstruction outputs to:', OUT)
        display(checkpoint_source_summary)
    else:
        raise RuntimeError('No reconstruction checkpoint evaluations ran.')
else:
    raise RuntimeError('RUN_STAGE1_RECON_EVAL is False. This notebook is configured to avoid silent no-op runs.')


RuntimeError: LEGACY_PUBMED_DATASETS is missing; rerun Stage 1 with AE_TRAINING_RECIPE="previous_good_compatible".

## Stage 2-3: Encoder Init And Contrastive Fine-Tuning

Initialize the brain encoder from the autoencoder encoder, then fine-tune for single-primary-text contrastive retrieval. Evaluation must report both directions separately:

```text
brain -> text
text -> brain
```

Main metrics: exact PMID paper recall AUC, exact PMID normalized-k AUC, MeSH normalized-k AUC, semantic paper-style AUC, macro retrieval normalized-k AUC, network accuracy, network macro AUC, and network term normalized-k AUC.

In [ ]:
RUN_STAGE3_CONTRASTIVE = True
CONTRASTIVE_CKPT = OUT / 'stage3_contrastive/checkpoints/last.pt'

# Stage 3 contrastive training using train_ale_cnn.py

def normalized_auc(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    if len(x) < 2: return float('nan')
    return float(np.trapz(y, x) / max(x[-1] - x[0], 1e-8))

def recall_curve(scores, positive_mask, k_values):
    order = torch.argsort(scores, dim=1, descending=True)
    curves = []
    for k in k_values:
        hit = positive_mask.gather(1, order[:, :k]).any(dim=1).float().mean().item()
        curves.append(hit)
    return curves

def bidirectional_retrieval_report(brain_emb, text_emb, positive_mask, k_values=None):
    brain_emb = torch.nn.functional.normalize(brain_emb.float(), dim=1)
    text_emb = torch.nn.functional.normalize(text_emb.float(), dim=1)
    scores_bt = brain_emb @ text_emb.T
    scores_tb = scores_bt.T
    if k_values is None:
        k_values = [1, 5, 10, 25, 50, 100, min(250, scores_bt.shape[1])]
    normalized_k = [k / scores_bt.shape[1] for k in k_values]
    bt = recall_curve(scores_bt, positive_mask.bool(), k_values)
    tb = recall_curve(scores_tb, positive_mask.T.bool(), k_values)
    mean_curve = [(a + b) / 2 for a, b in zip(bt, tb)]
    return {
        'k_values': k_values,
        'normalized_k_values': normalized_k,
        'brain_to_text_recall_curve': bt,
        'text_to_brain_recall_curve': tb,
        'mean_recall_curve': mean_curve,
        'brain_to_text_normalized_k_recall_curve_auc': normalized_auc(normalized_k, bt),
        'text_to_brain_normalized_k_recall_curve_auc': normalized_auc(normalized_k, tb),
        'normalized_k_recall_curve_auc': normalized_auc(normalized_k, mean_curve),
    }

if RUN_STAGE3_CONTRASTIVE:
    print('Training Stage 3 contrastive brain-text encoder...')
    print('Text embeddings will be automatically downloaded from HuggingFace.')

    # Import training components from train_ale_cnn.py
    import sys
    sys.path.insert(0, str(ROOT / 'experiments/3dcnn'))
    from train_ale_cnn import ALETrainer, build_model, build_dataset, which_device

    # Create args namespace for Stage 3 training
    import argparse
    stage3_dir = OUT / 'stage3_contrastive'
    stage3_ckpt_dir = stage3_dir / 'checkpoints'
    stage3_dir.mkdir(parents=True, exist_ok=True)
    stage3_ckpt_dir.mkdir(parents=True, exist_ok=True)

    args = argparse.Namespace(
        # Data
        mode='atlas_free',
        kernel_fwhm_mm=9.0,
        resolution_mm=4.0,
        crop_to_brain=True,
        normalize='max',
        no_clamp=False,
        cache_dtype='float16',
        cache_file=None,  # Will use default
        force_rebuild_cache=False,
        max_papers=None,
        seed=SEED,
        val_frac=0.1,
        test_frac=0.1,

        # Model architecture (use autoencoder's trained encoder)
        model='ale_3dcnn',
        base_channels=int(ae_cfg['model']['base_channels']),
        num_blocks=int(ae_cfg['model']['num_blocks']),
        blocks_per_stage=2,
        out_dim=int(ae_cfg['model']['latent_dim']),  # 384
        dropout=float(ae_cfg['model']['dropout']),
        norm=str(ae_cfg['model']['norm']),
        pooling=str(ae_cfg['model']['pooling']),
        use_dilation=False,
        multi_scale=False,
        global_context='none',
        mlp_hidden_dim=1024,

        # Encoder initialization from Stage 1
        encoder_init='autoencoder_pretrained',
        autoencoder_checkpoint=str(AE_CKPT),
        text_proj_init='pretrained_infonce',
        freeze_text_proj=False,

        # Training
        epochs=20 if RUN_MODE == 'full' else 3,
        batch_size=128,
        batch_size_auto=True,
        batch_size_candidates='1024,768,512,384,256,192,128,96,64,32,16',
        grad_accum_steps=1,
        lr_cnn=1e-4,
        lr_proj=1e-4,
        warmup_epochs=2,
        temperature=0.07,
        val_interval=1,
        early_stopping_patience=10,
        monitor_metric='paper_recall_curve_auc',

        # System
        device='auto',
        amp=True,
        pin_memory=DEVICE.type == 'cuda',
        num_workers=4 if DEVICE.type == 'cuda' else 0,
        persistent_workers=DEVICE.type == 'cuda',

        # Paths
        run_dir=str(stage3_dir),
        checkpoint_dir=str(stage3_ckpt_dir),
        build_cache_only=False,
        resume_from=None,

        # Eval
        train_sanity_n=0,
        eval_neurovlm_baseline=False,
    )

    # Build dataset (text embeddings downloaded automatically from HuggingFace)
    print(f'Building Stage 3 dataset (mode={args.mode})...')
    ds, train_ds, val_ds, test_ds, payload, preprocess_config = build_dataset(args)
    print(f'Dataset: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} shape={ds.input_shape}')

    brain_encoder, text_proj = build_model(args, ds.input_shape)
    device = which_device(args.device)
    print(f'Stage 3 using device: {device}')

    # Save config
    (stage3_dir / 'config.json').write_text(json.dumps(vars(args), indent=2, default=str))
    (stage3_dir / 'preprocessing_config.json').write_text(json.dumps({**payload['config'], 'metadata': payload['metadata']}, indent=2, default=str))

    # Train
    print(f'Starting Stage 3 contrastive training (epochs={args.epochs})...')
    trainer = ALETrainer(brain_encoder, text_proj, args, device)
    trainer.preflight_batch_size(train_ds)
    print(f'Selected batch_size={args.batch_size}')

    trainer.fit(train_ds, val_ds)
    trainer.restore_best()

    # Evaluate
    test_metrics, test_brain_emb, test_text_emb = trainer.evaluate(test_ds)
    print(f"Stage 3 TEST: paper_recall_auc={test_metrics['paper_recall_curve_auc']:.4f} "
          f"r@1={test_metrics['mean_recall@1']:.4f} r@5={test_metrics['mean_recall@5']:.4f}")

    # Save results
    (stage3_dir / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2))
    CONTRASTIVE_CKPT = stage3_ckpt_dir / 'last.pt'
    print(f'Stage 3 checkpoint: {CONTRASTIVE_CKPT}')

elif CONTRASTIVE_CKPT.exists():
    print(f'Using existing Stage 3 checkpoint: {CONTRASTIVE_CKPT}')
else:
    print('Stage 3 contrastive training skipped. Set RUN_STAGE3_CONTRASTIVE=True to train.')

# retrieval_report = bidirectional_retrieval_report(test_brain_emb, test_text_emb, test_positive_mask)
# json.dump(retrieval_report, open(OUT/'stage3_bidirectional_retrieval_curves.json','w'), indent=2)

## Stage 5: Held-Out Text-To-Brain Generation Evaluation

Default evaluation path:

```text
held-out test text -> SPECTER/SPECTER2 -> trained text-to-brain projection -> CNN decoder -> generated ALE volume
```

This is evaluation on test. Do not fit anything in this stage.

In [ ]:
from atlas_free_cnn.training.generation_baselines import global_mean_map, random_training_maps, nearest_neighbor_text_maps, category_average_maps, predict_category_average

def primary_text_table(rows, text_pairs, split):
    split_maps = rows[rows['split'] == split][['map_id','tensor_index','primary_text']]
    primary = text_pairs[(text_pairs['split'] == split) & (text_pairs['is_primary_text'])].copy()
    return split_maps.merge(primary[['map_id','text','term','category','source']], on='map_id', how='left')

def evaluate_generation_tensors(pred, target, batch_size=32):
    rows_out = []
    for start in range(0, len(target), batch_size):
        p = pred[start:start + batch_size].float()
        t = target[start:start + batch_size].float()
        m = generation_metrics(p, t, include_voxel_auroc=True)
        m['mae'] = mae(p, t)
        rows_out.append(m)
    return {k: float(np.nanmean([r[k] for r in rows_out])) for k in rows_out[0]}

def plot_random_generation_examples(true, generated, meta, out_png, out_dir, n=6):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    idxs = random.sample(range(len(true)), k=min(n, len(true)))
    fig, axes = plt.subplots(len(idxs), 9, figsize=(18, 2.2 * len(idxs)))
    if len(idxs) == 1: axes = np.expand_dims(axes, 0)
    for r, idx in enumerate(idxs):
        t = true[idx]; g = generated[idx]; diff = (g - t).abs()
        vmax = max(float(t.max()), float(g.max()), 1e-6)
        metric = generation_metrics(g.unsqueeze(0), t.unsqueeze(0), include_voxel_auroc=False)
        panels = _middle_slices(t) + _middle_slices(g) + _middle_slices(diff)
        titles = ['true axial','true coronal','true sagittal','gen axial','gen coronal','gen sagittal','diff axial','diff coronal','diff sagittal']
        for c, panel in enumerate(panels):
            axes[r, c].imshow(panel.T, origin='lower', cmap='magma', vmin=0, vmax=vmax if c < 6 else None)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].set_title(titles[c], fontsize=8)
        label = f"test\n{meta.iloc[idx].map_id}\nr={metric['spatial_corr']:.2f} d5={metric['top5_dice']:.2f}"
        axes[r, 0].set_ylabel(label, fontsize=8)
        fig.savefig(out_dir / f"test_{meta.iloc[idx].map_id}.png", dpi=160, bbox_inches='tight')
    fig.tight_layout(); fig.savefig(out_png, dpi=180, bbox_inches='tight')
    return str(out_png), str(out_dir)

test_primary = primary_text_table(rows, text_pairs, 'test')
train_primary = primary_text_table(rows, text_pairs, 'train')
test_target = volumes[test_primary.tensor_index.to_numpy()].float()
train_volumes = volumes[train_primary.tensor_index.to_numpy()].float()

# generated = generate_from_text(test_primary.text.tolist())  # SPECTER -> trained projection -> decoder
# gen_metrics = evaluate_generation_tensors(generated, test_target)
# gen_plot = plot_random_generation_examples(test_target, generated, test_primary, OUT/'random_generation_examples_test.png', OUT/'random_generation_examples_test')

# Baselines to compare against generated maps. These run even before a text-to-brain model is trained.
mean_pred = global_mean_map(train_volumes).unsqueeze(0).expand_as(test_target)
random_pred = random_training_maps(train_volumes, n=len(test_target), seed=SEED)
baselines = {'global_mean_map': mean_pred, 'random_training_map': random_pred}
baseline_metrics = {name: evaluate_generation_tensors(pred, test_target) for name, pred in baselines.items()}
json.dump(baseline_metrics, open(OUT / 'stage4_generation_baselines.json', 'w'), indent=2)
print('Saved generation baselines:', OUT / 'stage4_generation_baselines.json')

if 'generated' in globals():
    gen_metrics = evaluate_generation_tensors(generated, test_target)
    gen_plot = plot_random_generation_examples(test_target, generated, test_primary, OUT/'random_generation_examples_test.png', OUT/'random_generation_examples_test')
else:
    gen_metrics = None
    gen_plot = None
    print('No generated maps found yet. Run Stage 4 text-to-brain projection training first, create `generated`, then rerun this cell for generation metrics.')

baseline_metrics

## Stage 4: Train Text-To-Brain Projection Head

Only use train for fitting and val for model selection.

```text
SPECTER/SPECTER2 text -> text-to-brain projection head -> CNN latent -> frozen CNN decoder -> reconstructed ALE volume
```

Loss:

```text
latent_alignment_loss(text_z, brain_z)
+ mse_loss(raw_decoded, true_ALE)
```

Then evaluate the selected checkpoint once on test using Stage 5.

In [ ]:
# Download SPECTER text embedding cache from HuggingFace if not present locally
SPECTER_CACHE_PATH = CACHE / 'text_embeddings/specter_text_cache.pt'

if not SPECTER_CACHE_PATH.exists():
    print(f'Downloading specter_text_cache.pt from HuggingFace...')
    from huggingface_hub import hf_hub_download
    SPECTER_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    downloaded_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename='specter_text_cache.pt',
        repo_type='dataset'
    )
    # Copy to expected location
    import shutil
    shutil.copy(downloaded_path, SPECTER_CACHE_PATH)
    print(f'✓ Downloaded to {SPECTER_CACHE_PATH}')
else:
    print(f'✓ specter_text_cache.pt already exists at {SPECTER_CACHE_PATH}')


In [ ]:
from atlas_free_cnn.training.train_text_to_brain import train_from_config as train_text_to_brain

t2b_cfg = {
    'train_jsonl': str(SPLIT_JSONL['train']),
    'val_jsonl': str(SPLIT_JSONL['val']),
    'test_jsonl': str(SPLIT_JSONL['test']),
    'eval_jsonls': {'networks_test': str(CACHE / 'network_eval/network_test.jsonl')},
    'text_embedding_cache': str(CACHE / 'text_embeddings/specter_text_cache.pt'),
    'autoencoder_checkpoint': str(AE_CKPT),
    'target_shape': list(TARGET_SHAPE),
    'output_dir': str(OUT / 'stage4_text_to_brain'),
    'checkpoint_dir': str(OUT / 'stage4_text_to_brain/checkpoints'),
    'epochs': 10,
    'batch_size': int(ae_cfg['batch_size']),
    'preflight_batch_size': True,
    'batch_candidates': [4096, 3072, 2048, 1536, 1024, 768, 512],
    'lr': 1e-4,
    'progress': True,
    'model': dict(ae_cfg['model']),
    'text_projection_init': 'pretrained_text_infonce',
    'text_to_brain_projection': {'hidden_dim': 1536, 'depth': 4, 'dropout': 0.1},
    'prediction_activation': 'none',
    'loss': {'lambda_latent': 1.0, 'lambda_recon': 1.0, 'lambda_dice': 0.0, 'lambda_topk': 0.0, 'lambda_corr': 0.0},
    'weighted_recon': {'type': 'mse', 'alpha': 0.0, 'gamma': 1.0},
}

RUN_STAGE4_TEXT_TO_BRAIN = True
T2B_CKPT = OUT / 'stage4_text_to_brain/checkpoints/best_val_loss.pt'
if RUN_STAGE4_TEXT_TO_BRAIN:
    if not Path(t2b_cfg['text_embedding_cache']).exists():
        raise FileNotFoundError('Stage 4 needs cached SPECTER/SPECTER2 embeddings: ' + t2b_cfg['text_embedding_cache'])
    print('Training Stage 4 text-to-brain projection head.')
    t2b_result = train_text_to_brain(t2b_cfg)
    T2B_CKPT = Path(t2b_result['checkpoint_dir']) / 'best_val_loss.pt'
elif T2B_CKPT.exists():
    print('Using existing Stage 4 checkpoint:', T2B_CKPT)
else:
    print('Stage 4 text-to-brain projection not run. Set RUN_STAGE4_TEXT_TO_BRAIN=True after preparing text embeddings.')

T2B_CKPT

## Semantic Evaluation Of Generated Maps

Voxel metrics can understate useful semantic structure for sparse ALE maps. Also evaluate:

```text
text -> generated brain map -> brain encoder -> retrieve/rank text or terms
```

Report generated-map exact PMID retrieval diagnostic, semantic-neighborhood retrieval, MeSH term ranking, and network/term ranking when labels are available.

In [ ]:
from atlas_free_cnn.evaluation.evaluate_generation_semantic_retrieval import generated_map_retrieval_metrics

# generated_brain_emb = brain_encoder(generated.to(DEVICE)).detach().cpu()
# semantic_generation = generated_map_retrieval_metrics(generated_brain_emb, candidate_text_emb, positive_mask)
# json.dump(semantic_generation, open(OUT/'stage4_generated_semantic_retrieval.json','w'), indent=2)

## Final Summary Table

Save one final table with reconstruction, bidirectional retrieval, text-to-brain generation, baselines, semantic generation diagnostics, and qualitative plot paths.

In [ ]:

stage3_ran = bool(globals().get('RUN_STAGE3_CONTRASTIVE', False) and Path(globals().get('CONTRASTIVE_CKPT', '')).exists())
stage4_generated_maps_exist = bool('generated' in globals() and globals().get('gen_metrics') is not None)
stage4_ran = bool(globals().get('RUN_STAGE4_TEXT_TO_BRAIN', False) and Path(globals().get('T2B_CKPT', '')).exists())

summary_rows = []
summary_rows.extend([
    {'section': 'run_status', 'split': 'all', 'metric': 'run_mode', 'value': RUN_MODE},
    {'section': 'run_status', 'split': 'all', 'metric': 'data_mode', 'value': DATA_MODE},
    {'section': 'run_status', 'split': 'all', 'metric': 'ae_training_recipe', 'value': AE_TRAINING_RECIPE},
    {'section': 'run_status', 'split': 'all', 'metric': 'stage1_autoencoder', 'value': 'ran' if RUN_STAGE1_AUTOENCODER else 'loaded_existing_checkpoint'},
    {'section': 'run_status', 'split': 'all', 'metric': 'stage2_encoder_init', 'value': 'not_meaningful_until_stage3_runs'},
    {'section': 'run_status', 'split': 'all', 'metric': 'stage3_contrastive', 'value': 'ran' if stage3_ran else 'skipped'},
    {'section': 'run_status', 'split': 'all', 'metric': 'stage4_text_to_brain_projection', 'value': 'ran' if stage4_ran else 'skipped'},
    {'section': 'run_status', 'split': 'all', 'metric': 'stage5_text_to_brain_generation', 'value': 'generated_maps_exist' if stage4_generated_maps_exist else 'no_generated_maps_only_baselines'},
    {'section': 'architecture', 'split': 'stage1', 'metric': 'base_channels', 'value': int(ae_cfg['model']['base_channels'])},
    {'section': 'architecture', 'split': 'stage1', 'metric': 'num_blocks', 'value': int(ae_cfg['model']['num_blocks'])},
    {'section': 'architecture', 'split': 'stage1', 'metric': 'latent_dim', 'value': int(ae_cfg['model']['latent_dim'])},
    {'section': 'architecture', 'split': 'stage1', 'metric': 'epochs', 'value': int(ae_cfg['epochs'])},
    {'section': 'architecture', 'split': 'stage1', 'metric': 'batch_size', 'value': int(ae_cfg['batch_size'])},
])

if 'source_counts' in globals():
    for _, row in source_counts.iterrows():
        summary_rows.append({'section': 'source_counts', 'split': row['split'], 'metric': row['canonical_source'], 'value': int(row['n'])})
if 'checkpoint_source_summary' in globals() and not checkpoint_source_summary.empty:
    for _, row in checkpoint_source_summary.iterrows():
        for metric in ['reconstruction_mse', 'mse', 'mae', 'spatial_corr', 'top1_dice', 'top5_dice', 'top10_dice', 'voxel_auroc']:
            if metric in row and pd.notna(row[metric]):
                summary_rows.append({
                    'section': f"reconstruction:{row['checkpoint_label']}",
                    'split': f"{row['split']}:{row['source']}",
                    'metric': metric,
                    'value': float(row[metric]),
                })
if 'retrieval_report' in globals():
    for metric, value in retrieval_report.items():
        if isinstance(value, (int, float, np.floating)):
            summary_rows.append({'section': 'retrieval', 'split': 'test', 'metric': metric, 'value': float(value)})
if 'gen_metrics' in globals() and gen_metrics is not None:
    for metric, value in gen_metrics.items():
        summary_rows.append({'section': 'generation', 'split': 'test', 'metric': metric, 'value': float(value)})
if 'baseline_metrics' in globals():
    for name, metrics in baseline_metrics.items():
        for metric, value in metrics.items():
            summary_rows.append({'section': f'generation_baseline:{name}', 'split': 'test', 'metric': metric, 'value': float(value)})
if 'reconstruction_plot_paths' in globals():
    for label, paths in reconstruction_plot_paths.items():
        if paths is not None:
            summary_rows.append({'section': 'qualitative_reconstruction', 'split': label, 'metric': 'plot_png', 'value': paths[0]})
if 'gen_plot' in globals() and gen_plot is not None:
    summary_rows.append({'section': 'qualitative_generation', 'split': 'test', 'metric': 'plot_png', 'value': gen_plot[0]})



def _yes_no(value):
    if value is None:
        return 'unknown'
    return 'yes' if bool(value) else 'no'


def _mean_metric(df, *, checkpoint_contains=None, split=None, source=None, metric='spatial_corr'):
    if df is None or len(df) == 0 or metric not in df.columns:
        return None
    sub = df
    if checkpoint_contains is not None and 'checkpoint_label' in sub.columns:
        sub = sub[sub['checkpoint_label'].astype(str).str.contains(checkpoint_contains, regex=False)]
    if split is not None and 'split' in sub.columns:
        sub = sub[sub['split'].astype(str) == split]
    if source is not None and 'source' in sub.columns:
        sub = sub[sub['source'].astype(str) == source]
    if len(sub) == 0:
        return None
    value = pd.to_numeric(sub[metric], errors='coerce').mean()
    return None if pd.isna(value) else float(value)


def _ae_quality_label(df):
    # Prefer spatial correlation as the least surprising visual-reconstruction proxy;
    # fall back to top5 Dice if correlation is unavailable.
    if AE_TRAINING_RECIPE == 'previous_good_compatible':
        labels = ['new_previous_good_compatible_best_cnn_autoencoder', 'new_previous_good_compatible']
        split_name = 'legacy_pubmed_test' if DATA_MODE == 'pubmed_only' else 'mixed_test'
    else:
        labels = [f'current_{RUN_MODE}_{DATA_MODE}_best_val_loss', f'current_{RUN_MODE}_{DATA_MODE}']
        split_name = 'mixed_test' if DATA_MODE == 'mixed' else 'pubmed_only_test'
    corr = None
    for label in labels:
        corr = _mean_metric(df, checkpoint_contains=label, split=split_name, metric='spatial_corr')
        if corr is not None:
            break
    if corr is not None:
        if corr >= 0.35:
            return 'pass'
        if corr >= 0.15:
            return 'warn'
        return 'fail'
    d5 = None
    for label in labels:
        d5 = _mean_metric(df, checkpoint_contains=label, split=split_name, metric='top5_dice')
        if d5 is not None:
            break
    if d5 is None:
        return 'unknown'
    if d5 >= 0.20:
        return 'pass'
    if d5 >= 0.08:
        return 'warn'
    return 'fail'


def _full_beats_smoke(df):
    if df is None or len(df) == 0:
        return None
    full = _mean_metric(df, checkpoint_contains='current_full_', metric='spatial_corr')
    smoke = _mean_metric(df, checkpoint_contains='smoke', metric='spatial_corr')
    if full is None or smoke is None:
        return None
    return full > smoke


def _mixed_hurts_pubmed(df):
    if df is None or len(df) == 0:
        return None
    pubmed_only = _mean_metric(df, checkpoint_contains='current_full_', split='pubmed_only_test', source='pubmed', metric='spatial_corr')
    mixed_pubmed = _mean_metric(df, checkpoint_contains='current_full_', split='mixed_test', source='pubmed', metric='spatial_corr')
    if pubmed_only is None or mixed_pubmed is None:
        return None
    # Treat tiny differences as noise.
    return mixed_pubmed < (pubmed_only - 0.02)


_stage1_ckpt_dir = Path(ae_cfg['checkpoint_dir']) if 'ae_cfg' in globals() else None
stage1_ae_trained = bool(_stage1_ckpt_dir and any((_stage1_ckpt_dir / name).exists() for name in ['best_cnn_autoencoder.pt', 'best_val_loss.pt', 'best_top5_dice.pt', 'best_generation_top5_dice.pt', 'last.pt', 'last_cnn_autoencoder.pt']))
previous_known_good_found = bool(globals().get('PREVIOUS_GOOD_AE_CKPT') and Path(globals().get('PREVIOUS_GOOD_AE_CKPT')).exists())
all_recon_for_verdict = globals().get('all_recon_metrics', None)
run_verdict = {
    'Stage 1 AE trained': _yes_no(stage1_ae_trained),
    'Stage 1 AE quality': _ae_quality_label(all_recon_for_verdict),
    'Previous known-good checkpoint found': _yes_no(previous_known_good_found),
    'Full checkpoint beats smoke checkpoint': _yes_no(_full_beats_smoke(all_recon_for_verdict)),
    'Mixed data hurts PubMed reconstruction': _yes_no(_mixed_hurts_pubmed(all_recon_for_verdict)),
    'Stage 3 contrastive ran': _yes_no(stage3_ran),
    'Stage 5 generated maps exist': _yes_no(stage4_generated_maps_exist),
    'This run is complete full pipeline': _yes_no(stage1_ae_trained and stage3_ran and stage4_generated_maps_exist),
}
for metric, value in run_verdict.items():
    summary_rows.append({'section': 'run_verdict', 'split': 'all', 'metric': metric, 'value': value})

summary = pd.DataFrame(summary_rows)
if summary.empty:
    raise RuntimeError('No pipeline outputs were collected. Earlier training/evaluation cells did not run.')
summary_path = OUT / 'final_summary_table.csv'
summary.to_csv(summary_path, index=False)
print('FULL PIPELINE STATUS:')
for key, value in run_verdict.items():
    print(f'- {key}: {value}')
print('\nFinal status:')
print(f"- Stage 1 autoencoder: {'ran' if RUN_STAGE1_AUTOENCODER else 'loaded existing checkpoint'}")
print('- Stage 2 encoder init: not meaningful unless Stage 3 runs')
print(f"- Stage 3 contrastive: {'ran' if stage3_ran else 'skipped'}")
print(f"- Stage 5 text-to-brain generation: {'generated maps exist' if stage4_generated_maps_exist else 'no generated maps yet, only baselines'}")
print('This is not a completed full pipeline unless Stage 3 and Stage 4 generation evaluation actually run.')
print('Saved final summary:', summary_path)
display(summary)
